##Libraries and Data Imports

In [0]:
!pip install openpyxl
!pip install xlsxwriter
!pip install scikit-learn
!pip install statistics
!pip install rapidfuzz
!pip install networkx
!pip install tqdm
!pip install graphframes

In [0]:
import time
import io
import re
import pandas as pd
import numpy as np
import json
from pandas import json_normalize
from itertools import combinations

In [0]:
df_USBM = spark.read.csv(
   "dbfs:/mnt/analytics/riskcomp_corp/output/Test_Databricks/Organics Data Food/TJXUS_FOOD.csv",
   header=True,
   inferSchema=True
)

df_CA = spark.read.csv(
   "dbfs:/mnt/analytics/riskcomp_corp/output/Test_Databricks/Organics Data Food/TJXCA_FOOD.csv",
   header=True,
   inferSchema=True
)

df_EU = spark.read.csv(
   "dbfs:/mnt/analytics/riskcomp_corp/output/Test_Databricks/Organics Data Food/TJXE_FOOD.csv",
   header=True,
   inferSchema=True
)
df_lookup = spark.read.csv(
   "dbfs:/mnt/analytics/riskcomp_corp/output/Test_Databricks/Organics Data Food/LastOneYearPOData.csv",
   header=True,
   inferSchema=True
)

In [0]:
dbutils.fs.cp("dbfs:/mnt/analytics/riskcomp_corp/output/Test_Databricks/Organics Data Food/TJXEcom_FOOD.xlsx", "dbfs:/workfile/TJXEcom_FOOD.xlsx")

In [0]:
dbutils.fs.cp("dbfs:/mnt/analytics/riskcomp_corp/output/Test_Databricks/Organics Data Food/TJXSierra_FOOD.xlsx", "dbfs:/workfile/TJXSierra_FOOD.xlsx")

In [0]:
file_path = "/dbfs/workfile/TJXSierra_FOOD.xlsx"
df_Sierra = pd.read_excel(file_path, sheet_name='in')
print(df_Sierra.head())

In [0]:
file_path = "/dbfs/workfile/TJXEcom_FOOD.xlsx"
df_Ecom = pd.read_excel(file_path, sheet_name='in')
print(df_Ecom.head())

In [0]:
df_USBM_pd = df_USBM.toPandas()
df_USBM_copy = df_USBM_pd.copy()
df_USBM_copy.head(5)

In [0]:
# Calculate unique counts using pandas
unique_depts = df_USBM_copy['Dept'].nunique()
unique_vendor_styles = df_USBM_copy['Vendor Style'].nunique()
# Display results
print(f"Number of unique departments: {unique_depts}")
print(f"Number of unique vendor styles: {unique_vendor_styles}")
# Optional: Return as a dictionary
unique_counts = {
   "unique_departments": unique_depts,
   "unique_vendor_styles": unique_vendor_styles
}

In [0]:
df_CA_pd = df_CA.toPandas()
df_CA_copy = df_CA_pd.copy()
df_CA_copy.head(5)

In [0]:
# Calculate unique counts using pandas
unique_depts = df_CA_copy['Dept'].nunique()
unique_vendor_styles = df_CA_copy['Vendor Style'].nunique()
# Display results
print(f"Number of unique departments: {unique_depts}")
print(f"Number of unique vendor styles: {unique_vendor_styles}")
# Optional: Return as a dictionary
unique_counts = {
   "unique_departments": unique_depts,
   "unique_vendor_styles": unique_vendor_styles
}

In [0]:
df_EU_pd = df_EU.toPandas()
df_EU_copy = df_EU_pd.copy()
df_EU_copy.head(5)

In [0]:
# Calculate unique counts using pandas
unique_depts = df_EU_copy['Dept'].nunique()
unique_vendor_styles = df_EU_copy['Vendor Style'].nunique()
# Display results
print(f"Number of unique departments: {unique_depts}")
print(f"Number of unique vendor styles: {unique_vendor_styles}")
# Optional: Return as a dictionary
unique_counts = {
   "unique_departments": unique_depts,
   "unique_vendor_styles": unique_vendor_styles
}

In [0]:
df_lookup_pd = df_lookup.toPandas()
df_lookup_pd.columns

In [0]:
df_lookup_pd.columns, df_USBM_copy.columns

In [0]:
df_EU_copy.columns

In [0]:
display(df_lookup)

In [0]:
# Make a copy to keep the original intact

df_USBM_updated = df_USBM_copy.copy()

# Rename lookup columns to match USBM's column names

df_lookup_aligned = df_lookup_pd.rename(columns={

    'PO_NUM_MSTR': 'PO #',

    'PO_PRE_NUM': 'PO Pre #',

    'DEPT_NMBR': 'Dept',

    'VSN': 'Vendor Style',

    'CREATE_DATE': 'PO Date'   # <- assuming CREATE_DATE is your PO date column

})



# Keep only necessary columns from lookup

df_lookup_aligned = df_lookup_aligned[['PO #', 'PO Pre #', 'Dept', 'CON_CAN_DATE', 'Vendor Style', 'PO Date']]

# Ensure consistent casing and strip spaces for join keys

for col in ['PO #', 'PO Pre #', 'Dept', 'CON_CAN_DATE', 'Vendor Style']:

    df_lookup_aligned[col] = df_lookup_aligned[col].astype(str).str.strip().str.upper()

    df_USBM_updated[col] = df_USBM_updated[col].astype(str).str.strip().str.upper()

# Drop duplicates in lookup before merge

df_lookup_aligned = df_lookup_aligned.drop_duplicates(subset=['PO #', 'PO Pre #', 'Dept', 'CON_CAN_DATE', 'Vendor Style'])

# Perform left merge (keeps all rows in USBM)

df_USBM_updated = df_USBM_updated.merge(

    df_lookup_aligned,

    on=['PO #', 'PO Pre #', 'Dept', 'CON_CAN_DATE', 'Vendor Style'],

    how='left'

)

#  Final check

print(df_USBM_updated[['PO #', 'PO Pre #', 'Dept', 'Vendor Style', 'PO Date']].head())

print("Total filled PO Dates:", df_USBM_updated['PO Date'].notna().sum())
 

In [0]:
df_USBM_copy = df_USBM_updated

##Step 1: Standardize dataframes and Unification

In [0]:
import pandas as pd, numpy as np, re, unicodedata

# -------------------------------------------------

# PRE-FIX: align Lawson column name in US B&M file

# -------------------------------------------------

if "VEND_LAWSON" in df_USBM_copy.columns and "Master Vendor Number" not in df_USBM_copy.columns:

    df_USBM_copy = df_USBM_copy.rename(columns={"VEND_LAWSON": "Master Vendor Number"})

# ----------------------

# Helpers (no date parsing)

# ----------------------

def ensure_col(df, col):

    if col not in df.columns:

        df[col] = np.nan

def clean_vendor_name(x):

    if not isinstance(x, str): return ""

    x = unicodedata.normalize("NFKC", x).lower().strip()

    x = re.sub(r"[^\w\s&+]", " ", x)   # keep letters/digits/&/+

    return re.sub(r"\s+", " ", x).strip()

def clean_vsn(x):

    """Normalize separators/spaces ONLY; DO NOT strip or change suffix content."""

    if not isinstance(x, str): return ""

    x = unicodedata.normalize("NFKC", x).strip()

    x = x.replace(" ", "").replace("/", "-")

    x = re.sub(r"-{2,}", "-", x)       # collapse multiple dashes

    return x

def clean_sku(x):

    if not isinstance(x, str): return ""

    x = unicodedata.normalize("NFKC", x).strip()

    return re.sub(r"\s+", "", x)

def clean_desc(x):

    if not isinstance(x, str): return ""

    x = unicodedata.normalize("NFKC", x).lower()

    x = re.sub(r"[^\w\s]", " ", x)

    return re.sub(r"\s+", " ", x).strip()

def build_vendor_proxy(row):

    """

    Canonical vendor key for blocking/features:

      1) Master Vendor Number (Lawson) if present

      2) else Vendor Name (clean)

    (Vendor Number is intentionally NOT used here.)

    """

    mvn = row.get("Master Vendor Number", np.nan)

    if pd.notna(mvn) and str(mvn).strip() != "":

        return str(mvn).strip()

    namec = row.get("Vendor Name (clean)", "")

    return namec if isinstance(namec, str) else ""

def standardize_frame(df, source_label):

    df = df.copy()

    # Ensure the columns we might use exist (no renaming; helpers are added alongside)

    needed = [

        "Master Vendor Number", "Vendor Number",

        "Vendor Name", "Vendor Style", "SKU", "Product Description",

        "Banner", "Dept", "Department Description",

        "PO #", "PO Pre #",

        "Total Ordered Units", "Total Original Units", "Total Current Units",

        "CON_CAN_DATE", "PO Date",          

        "CHANNEL", "Import/Domestic",

        "Class Name", "Category Name",

    ]

    for c in needed:

        ensure_col(df, c)

    # Add fuzzy-ready helper fields (originals remain untouched)

    df["Vendor Name (clean)"]         = df["Vendor Name"].apply(clean_vendor_name)

    df["Vendor Style (clean)"]        = df["Vendor Style"].apply(clean_vsn)       # suffixes preserved

    df["TJX Style (clean)"]           = df["SKU"].apply(clean_sku)

    df["Product Description (clean)"] = df["Product Description"].apply(clean_desc)

    # Canonical vendor key

    df["VENDOR_PROXY"] = df.apply(build_vendor_proxy, axis=1)

    # Tag origin

    df["source_region"] = source_label

    return df

# ----------------------

# Apply to your five dataframes

# ----------------------

std_USBM   = standardize_frame(df_USBM_copy, "US_BM")

std_CA     = standardize_frame(df_CA_copy,   "CA_BM")

std_EU     = standardize_frame(df_EU_copy,   "EU_BM")

std_Ecom   = standardize_frame(df_Ecom,      "ECOM")

std_Sierra = standardize_frame(df_Sierra,    "SIERRA")

# ----------------------

# Combine for downstream ML steps

# ----------------------

df_all = pd.concat([std_USBM, std_CA, std_EU, std_Ecom, std_Sierra], ignore_index=True, sort=False)

# Optional cleanup: drop legacy fields if they exist

drop_cols = ["Class Number", "Category Number"]

df_all = df_all.drop(columns=[c for c in drop_cols if c in df_all.columns])

# Quick sanity checks

print("Rows by source_region:")

print(df_all["source_region"].value_counts(dropna=False))



print("\nHelper columns now available:",

      ["Vendor Name (clean)", "Vendor Style (clean)", "TJX Style (clean)",

       "Product Description (clean)", "VENDOR_PROXY", "Master Vendor Number", "Dept"])
 

In [0]:
df_all.columns

In [0]:
df_all['source_region'].unique()

In [0]:
df_all['PO Date']

In [0]:
# Convert to datetime
df_all['PO Date'] = pd.to_datetime(df_all['PO Date'], errors='coerce')
# Format as mm dd yyyy
df_all['PO Date'] = df_all['PO Date'].dt.strftime('%m-%d-%Y')

In [0]:
df_all['PO Date']

##Step 2A: Assign deterministic IDs to obvious (Lawson+VSN combo) umbrellas

In [0]:
import numpy as np

import pandas as pd

# =============================================

# STEP 1 — Prepare trimmed Lawson + VSN columns

# =============================================

df_all["__MVN_TRIM"] = (

    df_all["Master Vendor Number"].astype(str).str.strip()

)

df_all["__MVN_TRIM"] = df_all["__MVN_TRIM"].mask(

    df_all["__MVN_TRIM"].eq("") | df_all["__MVN_TRIM"].str.lower().isin(["nan", "none"])

)

df_all["__VSN_TRIM"] = (

    df_all["Vendor Style"].astype(str).str.strip()

)

df_all["__VSN_TRIM"] = df_all["__VSN_TRIM"].mask(

    df_all["__VSN_TRIM"].eq("") | df_all["__VSN_TRIM"].str.lower().isin(["nan", "none"])

)

# Valid rows: have both Lawson + VSN

valid_mask = df_all["__MVN_TRIM"].notna() & df_all["__VSN_TRIM"].notna()

# =============================================

# STEP 2 — Assign shared ID1 for valid combos

# =============================================

anchors = (

    df_all.loc[valid_mask, ["__MVN_TRIM", "__VSN_TRIM"]]

          .drop_duplicates()

          .reset_index(drop=True)

)

anchors["ID1"] = (anchors.index + 1).map(lambda i: f"ID1_{i:06d}")

df_all = df_all.merge(

    anchors,

    on=["__MVN_TRIM", "__VSN_TRIM"],

    how="left",

    validate="m:1"

)

# =============================================

# STEP 3 — Assign unique ID1 for records without Lawson/VSN

# =============================================

missing_mask = df_all["ID1"].isna()

if missing_mask.any():

    start_idx = anchors.shape[0] + 1

    n_missing = missing_mask.sum()

    fallback_ids = [

        f"ID1_{i:06d}" for i in range(start_idx, start_idx + n_missing)

    ]

    df_all.loc[missing_mask, "ID1"] = fallback_ids

# =============================================

# STEP 4 — Create clean subset + summary

# =============================================

df_good = df_all.copy()

print(" ID1 assignment complete")

print("Records total:", len(df_all))

print("Records with valid Lawson+VSN umbrella:", valid_mask.sum())

print("Records with fallback unique ID1:", missing_mask.sum())

print("Total unique ID1s:", df_good["ID1"].nunique())

# Optional quick preview

cols_preview = [

    c for c in [

        "ID1", "Master Vendor Number", "Vendor Style", "SKU",

        "Vendor Name", "Product Description", "source_region"

    ] if c in df_good.columns

]

try:

    display(df_good.head(20)[cols_preview])

except Exception as e:

    print("[display fallback]", e)

    print(df_good.head(20)[cols_preview].to_string(index=False))
 

In [0]:
df_all.to_csv("/dbfs/FileStore/EntireData.csv", index=False, encoding = "utf-8-sig")
# Getting the workspace URL

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")  


# File path relative to /dbfs/FileStore

file_name = "EntireData.csv"

download_url = f"https://{workspace_url}/files/{file_name}"

print("Download link:", download_url)
 

###Pre 100k selection

In [0]:
df_top_100k = df_good.copy()

In [0]:
df_top_100k.to_csv("/dbfs/FileStore/EntireSet.csv", index=False, encoding = "utf-8-sig")
# Getting the workspace URL

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")  


# File path relative to /dbfs/FileStore

file_name = "EntireSet.csv"

download_url = f"https://{workspace_url}/files/{file_name}"

print("Download link:", download_url)
 

In [0]:
print("Unique ID1s in 100k file:", df_top_100k["ID1"].nunique())

In [0]:
# ============================================================
# Assign permanent record ID (rec_id) to df_top_100k
# ============================================================
df_top_100k = df_top_100k.reset_index(drop=True)
df_top_100k["rec_id"] = np.arange(1, len(df_top_100k) + 1)
print(f" rec_id assigned to df_top_100k — total records: {len(df_top_100k):,}")

In [0]:
# ============================================

# STEP 1 — Feature Engineering Base (Simplified)

# ============================================

import pandas as pd

import numpy as np

import re

# Ensure df_top_100k exists

assert 'df_top_100k' in globals(), "df_top_100k not found in environment."



# --- Columns to keep ---

keep_cols = [

    "Import/Domestic",

    "SKU",

    "Vendor Name (clean)",

    "Vendor Style (clean)",

    "Product Description (clean)",

    "Department Description",

    "Export Country",

    "Buyer Name",

    "Banner",

    "Class Name",

    "Category Name",

    "ID1",

    "source_region",

    "rec_id"

]

# Subset safely (only existing columns)

df_fe = df_top_100k.loc[:, [c for c in keep_cols if c in df_top_100k.columns]].copy()


# Quick sanity check

print("Step 1 complete — feature-engineering dataframe ready")

print(f"Rows: {len(df_fe):,}")

print(f"Columns: {list(df_fe.columns)}")

# Optional preview

df_fe.head(10)
 

In [0]:
#  Keep all regions, but downsample US_BM for memory control
usbm_sample = (
   df_fe.query("source_region == 'US_BM'")
        .sample(200_000, random_state=42)  # adjust if needed
)
df_fe = pd.concat([
   usbm_sample,
   df_fe.query("source_region != 'US_BM'")
], ignore_index=True)
print("Rows by source_region after sampling:")
print(df_fe['source_region'].value_counts())

In [0]:
df_fe.columns

In [0]:
df_fe["ID1"].nunique()

###New Check done until here

###Correlation Analysis

In [0]:
# ============================================

# STEP 2 — Correlation Analysis (Fixed)

# ============================================

import pandas as pd

from sklearn.preprocessing import LabelEncoder

# Make a copy

df_corr = df_fe.copy()

# Encode categorical / text columns numerically

encoders = {}

for col in df_corr.columns:

    if df_corr[col].dtype == "object" or isinstance(df_corr[col].iloc[0], str):

        le = LabelEncoder()

        df_corr[col] = le.fit_transform(df_corr[col].astype(str).fillna("NA"))

        encoders[col] = le

print(f"Encoded {len(encoders)} categorical columns for correlation analysis")

# --- Compute correlation matrix safely ---


corr_matrix = df_corr.select_dtypes(include=["number"]).corr()

# --- Display correlation matrix ---

print("\n=== Correlation Matrix (rounded to 3 decimals) ===")

print(corr_matrix.round(3))

# --- Identify top correlated pairs (absolute correlation > threshold) ---

threshold = 0.7

high_corr_pairs = (

    corr_matrix.unstack()

    .reset_index()

    .rename(columns={"level_0": "Feature_1", "level_1": "Feature_2", 0: "Correlation"})

)

high_corr_pairs = high_corr_pairs[

    (high_corr_pairs["Feature_1"] != high_corr_pairs["Feature_2"]) &

    (high_corr_pairs["Correlation"].abs() > threshold)

]

print(f"\n=== Highly Correlated Pairs (|corr| > {threshold}) ===")

print(high_corr_pairs.sort_values("Correlation", ascending=False).head(20).to_string(index=False))
 

###Mutual Information Analysis

In [0]:
# ============================================

# STEP 2B — Mutual Information (feature ranking)

# ============================================

import numpy as np

import pandas as pd

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_selection import mutual_info_classif

# --- sanity ---

assert 'df_fe' in globals(), "df_fe not found. Build it first."

# Target & features

TARGET_COL = "ID1"

X_cols = [c for c in df_fe.columns if c != TARGET_COL]

# --- Encode target (ID1 has many classes; MI supports that) ---

y_le = LabelEncoder()

y = y_le.fit_transform(df_fe[TARGET_COL].astype(str))

# --- Encode features ---

X_enc = pd.DataFrame(index=df_fe.index)

discrete_mask = []   # all features will be treated as discrete for MI

encoders = {}

for col in X_cols:

    s = df_fe[col]

    # If non-numeric or dtype object -> label encode

    if s.dtype == 'object' or str(s.dtype).startswith('category'):

        le = LabelEncoder()

        X_enc[col] = le.fit_transform(s.astype(str).fillna("NA"))

        encoders[col] = le

        discrete_mask.append(True)

    else:

        # numeric column: coerce NaNs to a sentinel and cast to int if appropriate

        if s.isna().any():

            s = s.fillna(-1)

        # treat numeric as discrete here (IDs like SKU behave better as discrete)

        X_enc[col] = s.astype(int, errors='ignore')

        discrete_mask.append(True)

discrete_mask = np.array(discrete_mask, dtype=bool)

# --- Compute Mutual Information ---

# n_neighbors can be tuned; with all-discrete features, it's ignored internally.

mi_scores = mutual_info_classif(

    X_enc.values, y,

    discrete_features=discrete_mask,

    random_state=17

)

mi_df = (

    pd.DataFrame({"feature": X_cols, "mi_score": mi_scores})

      .sort_values("mi_score", ascending=False)

      .reset_index(drop=True)

)

# Optional: normalize MI to [0,1] for easier reading (relative scaling)

if mi_df["mi_score"].max() > 0:

    mi_df["mi_norm"] = mi_df["mi_score"] / mi_df["mi_score"].max()

else:

    mi_df["mi_norm"] = 0.0

# --- Display results ---

print(" Mutual Information computed against ID1.")

print("\nTop 20 features by MI:")

print(mi_df.head(20).to_string(index=False))

# Display the full table:

# display(mi_df)
 

###Chi Square Test

In [0]:
# ============================================

# STEP 2C — Chi-Square Test (feature relevance vs. group field ID1)

# ============================================

import pandas as pd

import numpy as np

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_selection import chi2

# --- Sanity check ---

assert 'df_fe' in globals(), "df_fe not found. Run Step 1 first."

TARGET_COL = "ID1"

X_cols = [c for c in df_fe.columns if c != TARGET_COL]

# --- Encode the target (ID1) ---

y_le = LabelEncoder()

y = y_le.fit_transform(df_fe[TARGET_COL].astype(str))

# --- Encode categorical features numerically ---

X_enc = pd.DataFrame(index=df_fe.index)

encoders = {}

for col in X_cols:

    s = df_fe[col]

    if s.dtype == "object" or str(s.dtype).startswith("category"):

        le = LabelEncoder()

        X_enc[col] = le.fit_transform(s.astype(str).fillna("NA"))

        encoders[col] = le

    else:

        X_enc[col] = s.fillna(-1).astype(float)

# --- Compute Chi-Square statistics ---

chi2_scores, p_values = chi2(X_enc, y)

# --- Build results DataFrame ---

chi2_df = (

    pd.DataFrame({

        "feature": X_cols,

        "chi2_score": chi2_scores,

        "p_value": p_values

    })

    .sort_values("chi2_score", ascending=False)

    .reset_index(drop=True)

)

# --- Add significance flag ---

chi2_df["significant"] = chi2_df["p_value"] < 0.05

# --- Display results ---

print(" Chi-Square Test complete (vs ID1 — similarity grouping field)")

print("\nTop 20 features by Chi² statistic:")

print(chi2_df.head(20).to_string(index=False))
 


###ANOVA Testing

In [0]:
# ============================================

# STEP — ANOVA F-test for feature importance vs ID1

# ============================================

from sklearn.preprocessing import LabelEncoder

from sklearn.feature_selection import f_classif

import pandas as pd

import numpy as np

# --- Prepare data ---

df_anova = df_fe.copy()

# Encode ID1 as target (numeric groups)

le_y = LabelEncoder()

y = le_y.fit_transform(df_anova["ID1"].astype(str))

# Encode all non-ID1 columns

X = pd.DataFrame()

encoders = {}

for col in df_anova.columns:

    if col != "ID1":

        le = LabelEncoder()

        X[col] = le.fit_transform(df_anova[col].astype(str))

        encoders[col] = le

# --- Perform ANOVA F-test ---

f_vals, p_vals = f_classif(X, y)

anova_df = pd.DataFrame({

    "feature": X.columns,

    "f_score": f_vals,

    "p_value": p_vals

})

# Add a boolean for significance (p < 0.05)

anova_df["significant"] = anova_df["p_value"] < 0.05

# --- Display results ---

print(" ANOVA F-test complete (vs ID1 — similarity grouping field)\n")

print("Top 20 features by F-statistic:\n")

print(anova_df.sort_values("f_score", ascending=False).head(20).to_string(index=False))
 

###Candidate Pair Generation

In [0]:
df_fe.columns

In [0]:
# ============================================================

# Candidate generation WITHOUT hierarchy (cross-banner allowed)

#  - Light product-level blocking + TF-IDF cosine

#    P1: Vendor Style (clean)  -> char 3-5 grams

#    P2: Vendor Name (clean)   -> word 1-2 grams

#    P3: Product Description   -> word 1-2 grams

#    P4: Combined Hierarchy (Dept + Class + Category) -> word 1-2 grams

# ============================================================

import numpy as np

import pandas as pd

from typing import Optional, Tuple

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import normalize

# ---------- tunables ----------

MAX_BLOCK_SIZE = 900           # cap per block before O(n^2)

MAX_PAIRS_PER_BLOCK = 50_000   # safety cap per block

T_VSN   = 0.85

T_VEND  = 0.80

T_DESC  = 0.78

T_HIER  = 0.80                 # threshold for hierarchy context

MIN_PREFIX_LEN = 3

# ---------- ensure required columns / rec_id ----------

need = [

    "Vendor Style (clean)",

    "Vendor Name (clean)",

    "Product Description (clean)",

    "Department Description",

    "Class Name",

    "Category Name"

]

for c in need:

    if c not in df_fe.columns:

        df_fe[c] = np.nan

if "rec_id" not in df_fe.columns:

    df_fe = df_fe.reset_index(drop=True)

    df_fe["rec_id"] = (df_fe.index + 1).astype("int64")

# ---------- small text helpers ----------

def _norm_w(s):

    if not isinstance(s, str): return ""

    return " ".join(s.lower().strip().split())

def _first_token(s):

    s = _norm_w(s)

    return s.split(" ", 1)[0] if s else ""

def _first_sig_word(s):

    """First word with >=3 letters/digits; good for desc-based blocking."""

    s = _norm_w(s)

    for w in s.split():

        if any(ch.isalnum() for ch in w) and len(w) >= 3:

            return w

    return ""

def _prefix(s, k=MIN_PREFIX_LEN):

    return (s[:k] if isinstance(s, str) else "") if k > 0 else ""

def _len_bucket(s):

    n = len(s) if isinstance(s, str) else 0

    if   n <= 4:  return "L0"

    elif n <= 7:  return "L1"

    elif n <= 10: return "L2"

    elif n <= 14: return "L3"

    else:         return "L4"

# ---------- TF-IDF builder ----------

def _tfidf_matrix(texts: pd.Series, analyzer: str, ngram_range: Tuple[int, int],

                  min_df: int = 1, max_features: Optional[int] = 200_000):

    texts = texts.fillna("").astype(str)

    if (texts.str.len() == 0).all():

        return None, None, False

    try:

        vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range,

                              min_df=min_df, max_features=max_features)

        X = vec.fit_transform(texts.tolist())

        if X.shape[1] == 0:

            return None, None, False

        X = normalize(X, norm="l2", copy=False)

        return X, vec, True

    except ValueError:

        return None, None, False

def _cosine_pairs_from_X(X, ids: np.ndarray, threshold: float,

                         signal: str, max_pairs: Optional[int] = None) -> pd.DataFrame:

    n = X.shape[0]

    if n < 2:

        return pd.DataFrame(columns=["rec_id_A","rec_id_B","score","signal","block_key"])

    Xt = X.T

    A, B, S = [], [], []

    for i in range(n):

        sims = X[i].dot(Xt).toarray().ravel()

        sims[i] = 0.0

        keep = np.where(sims >= threshold)[0]

        keep = keep[keep > i]

        if keep.size:

            A.extend([ids[i]] * len(keep))

            B.extend(ids[keep])

            S.extend(sims[keep])

    if not S:

        return pd.DataFrame(columns=["rec_id_A","rec_id_B","score","signal","block_key"])

    dfp = pd.DataFrame({"rec_id_A":A, "rec_id_B":B, "score":S, "signal":signal})

    if max_pairs is not None and len(dfp) > max_pairs:

        topk = np.argpartition(-dfp["score"].values, max_pairs-1)[:max_pairs]

        dfp = dfp.iloc[topk]

    return dfp

pairs_all = []

# ============================================================

# PASS P1 — Vendor Style (clean): char 3–5 grams

# ============================================================

vsn_src = df_fe["Vendor Style (clean)"].fillna("").astype(str).str.strip()

blk_key_vsn = vsn_src.apply(lambda s: f"{_prefix(s.upper(), MIN_PREFIX_LEN)}|{_len_bucket(s)}")

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": vsn_src, "bkey": blk_key_vsn})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P1] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2: continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=23)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="char", ngram_range=(3,5))

    if not ok: continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_VSN, "VSN_3-5")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P1] pairs: {cnt_pairs:,}")

# ============================================================

# PASS P2 — Vendor Name (clean): word 1–2 grams

# ============================================================

vend_src = df_fe["Vendor Name (clean)"].map(_norm_w)

vend_tok  = vend_src.map(_first_token)

blk_key_vend = vend_tok.map(lambda t: _prefix(t, MIN_PREFIX_LEN))

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": vend_src, "bkey": blk_key_vend})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P2] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2: continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=24)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="word", ngram_range=(1,2), min_df=2)

    if not ok: continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_VEND, "VENDOR_1-2")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P2] pairs: {cnt_pairs:,}")

# ============================================================

# PASS P3 — Product Description (clean): word 1–2 grams

# ============================================================

desc_src = df_fe["Product Description (clean)"].map(_norm_w)

first_sig = desc_src.map(_first_sig_word)

blk_key_desc = first_sig.map(lambda t: _prefix(t, MIN_PREFIX_LEN))

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": desc_src, "bkey": blk_key_desc})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P3] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2: continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=25)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="word", ngram_range=(1,2), min_df=2)

    if not ok: continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_DESC, "DESC_1-2")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P3] pairs: {cnt_pairs:,}")

# ============================================================

# PASS P4 — Combined Hierarchy (Dept + Class + Category): word 1–2 grams

# ============================================================

df_fe["Hierarchy (combined)"] = (

    df_fe[["Department Description","Class Name","Category Name"]]

    .astype(str)

    .apply(lambda x: " ".join(_norm_w(v) for v in x if isinstance(v, str)), axis=1)

)

hier_src = df_fe["Hierarchy (combined)"].map(_norm_w)

blk_key_hier = hier_src.map(lambda t: _prefix(t, MIN_PREFIX_LEN))

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": hier_src, "bkey": blk_key_hier})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P4] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2: continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=26)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="word", ngram_range=(1,2), min_df=2)

    if not ok: continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_HIER, "HIER_1-2")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P4] pairs: {cnt_pairs:,}")

# ---------- finalize ----------

if pairs_all:

    pairs_cand = (

        pd.concat(pairs_all, ignore_index=True)

          .drop_duplicates(["rec_id_A","rec_id_B","signal"], keep="first")

          .reset_index(drop=True)

    )

    print("Candidate generation complete.")

    print(f"Total candidate pairs: {len(pairs_cand):,}")

    print("Signals distribution:\n", pairs_cand["signal"].value_counts(dropna=False).head(10))

else:

    pairs_cand = pd.DataFrame(columns=["rec_id_A","rec_id_B","score","signal","block_key"])

    print("No pairs generated — try adjusting thresholds or block size.")
 

In [0]:
#OLD CODE HEREEEE WORKS THO ============================================================

# Candidate generation WITHOUT hierarchy (cross-banner allowed)

#  - Light product-level blocking + TF-IDF cosine

#    P1: Vendor Style (clean)  -> char 3-5 grams

#    P2: Vendor Name (clean)   -> word 1-2 grams

#    P3: Product Description   -> word 1-2 grams

# ============================================================

import numpy as np

import pandas as pd

from typing import Optional, Tuple

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import normalize

# ---------- tunables ----------

MAX_BLOCK_SIZE = 1000           # cap per block before O(n^2)

MAX_PAIRS_PER_BLOCK = 50_000   # safety cap per block

T_VSN  = 0.85                  # thresholds per signal

T_VEND = 0.70

T_DESC = 0.75

MIN_PREFIX_LEN = 3             # prefix length for light blocking

# ---------- ensure required columns / rec_id ----------

need = ["Vendor Style (clean)", "Vendor Name (clean)", "Product Description (clean)"]

for c in need:

    if c not in df_fe.columns:

        df_fe[c] = np.nan

if "rec_id" not in df_fe.columns:

    df_fe = df_fe.reset_index(drop=True)

    df_fe["rec_id"] = (df_fe.index + 1).astype("int64")

# ---------- small text helpers ----------

def _norm_w(s):

    if not isinstance(s, str): return ""

    return " ".join(s.lower().strip().split())

def _first_token(s):

    s = _norm_w(s)

    return s.split(" ", 1)[0] if s else ""

def _first_sig_word(s):

    """First word with >=3 letters/digits; good for desc-based blocking."""

    s = _norm_w(s)

    for w in s.split():

        if any(ch.isalnum() for ch in w) and len(w) >= 3:

            return w

    return ""

def _prefix(s, k=MIN_PREFIX_LEN):

    return (s[:k] if isinstance(s, str) else "") if k > 0 else ""

def _len_bucket(s):

    n = len(s) if isinstance(s, str) else 0

    # 0-4, 5-7, 8-10, 11-14, 15+

    if   n <= 4:  return "L0"

    elif n <= 7:  return "L1"

    elif n <= 10: return "L2"

    elif n <= 14: return "L3"

    else:         return "L4"

# ---------- TF-IDF builder ----------

def _tfidf_matrix(texts: pd.Series, analyzer: str, ngram_range: Tuple[int, int],

                  min_df: int = 1, max_features: Optional[int] = 200_000):

    texts = texts.fillna("").astype(str)

    if (texts.str.len() == 0).all():

        return None, None, False

    try:

        vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range,

                              min_df=min_df, max_features=max_features)

        X = vec.fit_transform(texts.tolist())

        if X.shape[1] == 0:

            return None, None, False

        X = normalize(X, norm="l2", copy=False)

        return X, vec, True

    except ValueError:

        return None, None, False

def _cosine_pairs_from_X(X, ids: np.ndarray, threshold: float,

                         signal: str, max_pairs: Optional[int] = None) -> pd.DataFrame:

    """Row-wise cosine against all rows (upper triangle only)."""

    n = X.shape[0]

    if n < 2:

        return pd.DataFrame(columns=["rec_id_A","rec_id_B","score","signal","block_key"])

    Xt = X.T

    A, B, S = [], [], []

    for i in range(n):

        sims = X[i].dot(Xt).toarray().ravel()

        sims[i] = 0.0

        keep = np.where(sims >= threshold)[0]

        keep = keep[keep > i]

        if keep.size:

            A.extend([ids[i]] * len(keep))

            B.extend(ids[keep])

            S.extend(sims[keep])

    if not S:

        return pd.DataFrame(columns=["rec_id_A","rec_id_B","score","signal","block_key"])

    dfp = pd.DataFrame({"rec_id_A":A, "rec_id_B":B, "score":S, "signal":signal})

    if max_pairs is not None and len(dfp) > max_pairs:

        topk = np.argpartition(-dfp["score"].values, max_pairs-1)[:max_pairs]

        dfp = dfp.iloc[topk]

    return dfp

pairs_all = []

# ============================================================

# PASS P1 — Vendor Style (clean): char 3–5 grams

#   Block key: prefix(3) + length bucket   (cross-banner!)

# ============================================================

vsn_src = df_fe["Vendor Style (clean)"].fillna("").astype(str).str.strip()

blk_key_vsn = vsn_src.apply(lambda s: f"{_prefix(s.upper(), MIN_PREFIX_LEN)}|{_len_bucket(s)}")

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": vsn_src, "bkey": blk_key_vsn})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P1] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2:

        continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=23)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="char", ngram_range=(3,5))

    if not ok:

        continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_VSN, "VSN_3-5")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P1] pairs: {cnt_pairs:,}")

# ============================================================

# PASS P2 — Vendor Name (clean): word 1–2 grams

#   Block key: first_token prefix(3)

# ============================================================

vend_src = df_fe["Vendor Name (clean)"].map(_norm_w)

vend_tok  = vend_src.map(_first_token)

blk_key_vend = vend_tok.map(lambda t: _prefix(t, MIN_PREFIX_LEN))

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": vend_src, "bkey": blk_key_vend})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P2] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2:

        continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=24)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="word", ngram_range=(1,2), min_df=2)

    if not ok:

        continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_VEND, "VENDOR_1-2")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P2] pairs: {cnt_pairs:,}")

# ============================================================

# PASS P3 — Product Description (clean): word 1–2 grams

#   Block key: first significant word prefix(3)

# ============================================================

desc_src = df_fe["Product Description (clean)"].map(_norm_w)

first_sig = desc_src.map(_first_sig_word)

blk_key_desc = first_sig.map(lambda t: _prefix(t, MIN_PREFIX_LEN))

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": desc_src, "bkey": blk_key_desc})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P3] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2:

        continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=25)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="word", ngram_range=(1,2), min_df=2)

    if not ok:

        continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_DESC, "DESC_1-2")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P3] pairs: {cnt_pairs:,}")

# ---------- finalize ----------

if pairs_all:

    pairs_cand = (

        pd.concat(pairs_all, ignore_index=True)

          .drop_duplicates(["rec_id_A","rec_id_B","signal"], keep="first")

          .reset_index(drop=True)

    )

    print(" Candidate generation complete.")

    print(f"Total candidate pairs: {len(pairs_cand):,}")

    print("Signals distribution:\n", pairs_cand["signal"].value_counts(dropna=False).head(10))

    # Optional peek with context

    peek_cols = ["rec_id","Vendor Style (clean)","Vendor Name (clean)","Product Description (clean)","SKU","ID1","source_region"]

    peek = (

        pairs_cand

        .merge(df_fe[peek_cols], left_on="rec_id_A", right_on="rec_id", how="left")

        .rename(columns={

            "Vendor Style (clean)":"VSN_A",

            "Vendor Name (clean)":"Vendor_A",

            "Product Description (clean)":"Desc_A",

            "SKU":"SKU_A","ID1":"ID1_A","source_region":"REG_A"

        }).drop(columns=["rec_id"])

        .merge(df_fe[peek_cols], left_on="rec_id_B", right_on="rec_id", how="left")

        .rename(columns={

            "Vendor Style (clean)":"VSN_B",

            "Vendor Name (clean)":"Vendor_B",

            "Product Description (clean)":"Desc_B",

            "SKU":"SKU_B","ID1":"ID1_B","source_region":"REG_B"

        }).drop(columns=["rec_id"])

        .sort_values("score", ascending=False)

        .head(20)

    )

    # display(peek)  

else:

    pairs_cand = pd.DataFrame(columns=["rec_id_A","rec_id_B","score","signal","block_key"])

    print(" No pairs generated — try lowering thresholds (e.g., T_VSN=0.82, T_VEND=0.78, T_DESC=0.75) or increasing MAX_BLOCK_SIZE.")
 

In [0]:
#NEW CODE WITH SPARK PROCESSING + ADLS WRITE/READ ============================================================

# Candidate generation WITHOUT hierarchy (cross-banner allowed)

#  - Light product-level blocking + TF-IDF cosine

#    P1: Vendor Style (clean)  -> char 3-5 grams

#    P2: Vendor Name (clean)   -> word 1-2 grams

#    P3: Product Description   -> word 1-2 grams

# ============================================================

import numpy as np

import pandas as pd

from typing import Optional, Tuple

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import normalize

from pyspark.sql import functions as F

import uuid

# ---------- tunables ----------

MAX_BLOCK_SIZE = 1000           # cap per block before O(n^2)

MAX_PAIRS_PER_BLOCK = 50_000   # safety cap per block

T_VSN  = 0.85                  # thresholds per signal

T_VEND = 0.70

T_DESC = 0.75

MIN_PREFIX_LEN = 3             # prefix length for light blocking

# ---------- ensure required columns / rec_id ----------

need = ["Vendor Style (clean)", "Vendor Name (clean)", "Product Description (clean)"]

for c in need:

    if c not in df_fe.columns:

        df_fe[c] = np.nan

if "rec_id" not in df_fe.columns:

    df_fe = df_fe.reset_index(drop=True)

    df_fe["rec_id"] = (df_fe.index + 1).astype("int64")

# ---------- small text helpers ----------

def _norm_w(s):

    if not isinstance(s, str): return ""

    return " ".join(s.lower().strip().split())

def _first_token(s):

    s = _norm_w(s)

    return s.split(" ", 1)[0] if s else ""

def _first_sig_word(s):

    """First word with >=3 letters/digits; good for desc-based blocking."""

    s = _norm_w(s)

    for w in s.split():

        if any(ch.isalnum() for ch in w) and len(w) >= 3:

            return w

    return ""

def _prefix(s, k=MIN_PREFIX_LEN):

    return (s[:k] if isinstance(s, str) else "") if k > 0 else ""

def _len_bucket(s):

    n = len(s) if isinstance(s, str) else 0

    # 0-4, 5-7, 8-10, 11-14, 15+

    if   n <= 4:  return "L0"

    elif n <= 7:  return "L1"

    elif n <= 10: return "L2"

    elif n <= 14: return "L3"

    else:         return "L4"

# ---------- TF-IDF builder ----------

def _tfidf_matrix(texts: pd.Series, analyzer: str, ngram_range: Tuple[int, int],

                  min_df: int = 1, max_features: Optional[int] = 200_000):

    texts = texts.fillna("").astype(str)

    if (texts.str.len() == 0).all():

        return None, None, False

    try:

        vec = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram_range,

                              min_df=min_df, max_features=max_features)

        X = vec.fit_transform(texts.tolist())

        if X.shape[1] == 0:

            return None, None, False

        X = normalize(X, norm="l2", copy=False)

        return X, vec, True

    except ValueError:

        return None, None, False

def _cosine_pairs_from_X(X, ids: np.ndarray, threshold: float,

                         signal: str, max_pairs: Optional[int] = None) -> pd.DataFrame:

    """Row-wise cosine against all rows (upper triangle only)."""

    n = X.shape[0]

    if n < 2:

        return pd.DataFrame(columns=["rec_id_A","rec_id_B","score","signal","block_key"])

    Xt = X.T

    A, B, S = [], [], []

    for i in range(n):

        sims = X[i].dot(Xt).toarray().ravel()

        sims[i] = 0.0

        keep = np.where(sims >= threshold)[0]

        keep = keep[keep > i]

        if keep.size:

            A.extend([ids[i]] * len(keep))

            B.extend(ids[keep])

            S.extend(sims[keep])

    if not S:

        return pd.DataFrame(columns=["rec_id_A","rec_id_B","score","signal","block_key"])

    dfp = pd.DataFrame({"rec_id_A":A, "rec_id_B":B, "score":S, "signal":signal})

    if max_pairs is not None and len(dfp) > max_pairs:

        topk = np.argpartition(-dfp["score"].values, max_pairs-1)[:max_pairs]

        dfp = dfp.iloc[topk]

    return dfp

pairs_all = []

# ============================================================

# PASS P1 — Vendor Style (clean): char 3–5 grams

#   Block key: prefix(3) + length bucket   (cross-banner!)

# ============================================================

vsn_src = df_fe["Vendor Style (clean)"].fillna("").astype(str).str.strip()

blk_key_vsn = vsn_src.apply(lambda s: f"{_prefix(s.upper(), MIN_PREFIX_LEN)}|{_len_bucket(s)}")

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": vsn_src, "bkey": blk_key_vsn})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P1] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2:

        continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=23)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="char", ngram_range=(3,5))

    if not ok:

        continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_VSN, "VSN_3-5")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P1] pairs: {cnt_pairs:,}")

# ============================================================

# PASS P2 — Vendor Name (clean): word 1–2 grams

#   Block key: first_token prefix(3)

# ============================================================

vend_src = df_fe["Vendor Name (clean)"].map(_norm_w)

vend_tok  = vend_src.map(_first_token)

blk_key_vend = vend_tok.map(lambda t: _prefix(t, MIN_PREFIX_LEN))

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": vend_src, "bkey": blk_key_vend})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P2] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2:

        continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=24)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="word", ngram_range=(1,2), min_df=2)

    if not ok:

        continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_VEND, "VENDOR_1-2")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P2] pairs: {cnt_pairs:,}")

# ============================================================

# PASS P3 — Product Description (clean): word 1–2 grams

#   Block key: first significant word prefix(3)

# ============================================================

desc_src = df_fe["Product Description (clean)"].map(_norm_w)

first_sig = desc_src.map(_first_sig_word)

blk_key_desc = first_sig.map(lambda t: _prefix(t, MIN_PREFIX_LEN))

df_tmp = pd.DataFrame({"rec_id": df_fe["rec_id"], "text": desc_src, "bkey": blk_key_desc})

bcounts = df_tmp["bkey"].value_counts()

print(f"[P3] blocks: {len(bcounts)} (median size {int(bcounts.median())})")

cnt_pairs = 0

for key, g in df_tmp.groupby("bkey", sort=False):

    if len(g) < 2:

        continue

    if len(g) > MAX_BLOCK_SIZE:

        g = g.sample(MAX_BLOCK_SIZE, random_state=25)

    X, _, ok = _tfidf_matrix(g["text"], analyzer="word", ngram_range=(1,2), min_df=2)

    if not ok:

        continue

    dfp = _cosine_pairs_from_X(X, g["rec_id"].to_numpy(), T_DESC, "DESC_1-2")

    if not dfp.empty:

        dfp["block_key"] = key

        pairs_all.append(dfp)

        cnt_pairs += len(dfp)

print(f"[P3] pairs: {cnt_pairs:,}")

# ---------- finalize with SPARK + ADLS WRITE/READ ----------

if pairs_all:

    import time

    start_time = time.time()

    print("Converting pairs to Spark DataFrame with ADLS write/read...")

    # Write to ADLS mount location (following your org's pattern)

    temp_path = f"dbfs:/mnt/analytics/riskcomp_corp/output/temp_pairs_{uuid.uuid4().hex}"

    # Process and write pairs_all in chunks

    CHUNK_SIZE = 10

    for i in range(0, len(pairs_all), CHUNK_SIZE):

        chunk = pairs_all[i:i+CHUNK_SIZE]

        pairs_chunk = pd.concat(chunk, ignore_index=True)

        spark_df_chunk = spark.createDataFrame(pairs_chunk)

        # Write in append mode to ADLS

        if i == 0:

            spark_df_chunk.write.mode("overwrite").parquet(temp_path)

        else:

            spark_df_chunk.write.mode("append").parquet(temp_path)

        print(f"  Written chunk {i//CHUNK_SIZE + 1}/{(len(pairs_all)-1)//CHUNK_SIZE + 1}")

        # Free memory

        del pairs_chunk, spark_df_chunk

    write_time = time.time()

    print(f"Write phase completed in {(write_time - start_time)/60:.2f} minutes")

    # Read back as single DataFrame - much faster!

    print("Reading consolidated pairs from ADLS...")

    pairs_cand_spark = spark.read.parquet(temp_path)

    read_time = time.time()

    print(f"Read phase completed in {(read_time - write_time)/60:.2f} minutes")

    # Deduplicate in Spark

    print("Deduplicating pairs...")

    pairs_cand_spark = pairs_cand_spark.dropDuplicates(["rec_id_A", "rec_id_B", "signal"])

    # Cache for reuse

    pairs_cand_spark.cache()

    total_pairs = pairs_cand_spark.count()

    dedup_time = time.time()

    print(f"Deduplication completed in {(dedup_time - read_time)/60:.2f} minutes")

    print(" Candidate generation complete.")

    print(f"Total candidate pairs: {total_pairs:,}")

    print(f"TOTAL TIME: {(dedup_time - start_time)/60:.2f} minutes")

    # Get signal distribution

    signal_dist = pairs_cand_spark.groupBy("signal").count().orderBy(F.desc("count"))

    print("Signals distribution:")

    signal_dist.show(10, truncate=False)

    # Optional peek with context using Spark

    peek_cols = ["rec_id","Vendor Style (clean)","Vendor Name (clean)","Product Description (clean)","SKU","ID1","source_region"]

    # Convert df_fe to Spark

    print("Converting df_fe to Spark for joins...")

    df_fe_spark = spark.createDataFrame(df_fe[peek_cols])

    df_fe_spark.cache()

    # Join for side A

    peek_spark = (

        pairs_cand_spark

        .join(df_fe_spark.alias("a"), 

              pairs_cand_spark.rec_id_A == F.col("a.rec_id"), 

              "left")

        .select(

            "rec_id_A", "rec_id_B", "score", "signal", "block_key",

            F.col("a.`Vendor Style (clean)`").alias("VSN_A"),

            F.col("a.`Vendor Name (clean)`").alias("Vendor_A"),

            F.col("a.`Product Description (clean)`").alias("Desc_A"),

            F.col("a.SKU").alias("SKU_A"),

            F.col("a.ID1").alias("ID1_A"),

            F.col("a.source_region").alias("REG_A")

        )

    )

    # Join for side B

    peek_spark = (

        peek_spark

        .join(df_fe_spark.alias("b"), 

              peek_spark.rec_id_B == F.col("b.rec_id"), 

              "left")

        .select(

            "rec_id_A", "rec_id_B", "score", "signal", "block_key",

            "VSN_A", "Vendor_A", "Desc_A", "SKU_A", "ID1_A", "REG_A",

            F.col("b.`Vendor Style (clean)`").alias("VSN_B"),

            F.col("b.`Vendor Name (clean)`").alias("Vendor_B"),

            F.col("b.`Product Description (clean)`").alias("Desc_B"),

            F.col("b.SKU").alias("SKU_B"),

            F.col("b.ID1").alias("ID1_B"),

            F.col("b.source_region").alias("REG_B")

        )

        .orderBy(F.desc("score"))

        .limit(20)

    )

    # Show the peek

    print("\nTop 20 candidate pairs:")

    peek_spark.show(20, truncate=False)

    # Clean up temp files from ADLS

    print("Cleaning up temporary files...")

    dbutils.fs.rm(temp_path, recurse=True)

    # If you need pairs_cand as a variable for downstream processing

    pairs_cand = pairs_cand_spark

    print("\nNote: pairs_cand is now a Spark DataFrame.")

else:

    schema = "rec_id_A LONG, rec_id_B LONG, score DOUBLE, signal STRING, block_key STRING"

    pairs_cand = spark.createDataFrame([], schema)

    print(" No pairs generated — try lowering thresholds (e.g., T_VSN=0.82, T_VEND=0.78, T_DESC=0.75) or increasing MAX_BLOCK_SIZE.")
 

### Weak Supervision Labeling

In [0]:
# ================================

# Weak supervision labels for pairs_cand

# ================================

import numpy as np

import pandas as pd

import re, unicodedata

# ---------- Config ----------

# If pairs_cand is huge, label in chunks to keep memory under control

MAX_ROWS_TO_LABEL   = None     # cap rows processed (set None to do all)

CHUNK_SIZE          = 250_000       # per-chunk processing

VENDOR_POS_JACC     = 0.70          # vendor-name strong match

DESC_POS_JACC       = 0.70          # description strong match

VENDOR_NEG_JACC     = 0.20          # vendor-name clearly different

DESC_NEG_JACC       = 0.20          # description clearly different

# ---------- Safety checks ----------

req_pairs_cols = {"rec_id_A", "rec_id_B", "signal", "score"}

missing = req_pairs_cols.difference(pairs_cand.columns)

if missing:

    raise ValueError(f"pairs_cand missing required columns: {missing}")

# df_fe must have these columns keyed by rec_id

need_cols = [

    "rec_id",

    "Vendor Style (clean)",

    "Vendor Name (clean)",

    "Product Description (clean)",

    "Department Description",

    "Class Name",

    "Category Name",

    "ID1",

]

for c in need_cols:

    if c not in df_fe.columns:

        df_fe[c] = np.nan

# ---------- Light normalization + tokenization helpers ----------

_word = re.compile(r"[A-Za-z0-9]+")

def _canon(s):

    if not isinstance(s, str): return ""

    s = unicodedata.normalize("NFKC", s).lower().strip()

    return re.sub(r"\s+", " ", s)

def _tokens(s):

    if not isinstance(s, str): return frozenset()

    return frozenset(_word.findall(s.lower()))

def jaccard(a_tokens, b_tokens):

    if not a_tokens or not b_tokens:

        return 0.0

    inter = len(a_tokens & b_tokens)

    if inter == 0:

        return 0.0

    union = len(a_tokens | b_tokens)

    return inter / union

# ---------- Precompute lookup dicts keyed by rec_id ----------

fe_key = df_fe.set_index("rec_id")

# exact strings (already cleaned in your frame)

VSN_map    = fe_key["Vendor Style (clean)"].fillna("").astype(str).to_dict()

Dept_map   = fe_key["Department Description"].fillna("").astype(str).map(_canon).to_dict()

Class_map  = fe_key["Class Name"].fillna("").astype(str).map(_canon).to_dict()

Cat_map    = fe_key["Category Name"].fillna("").astype(str).map(_canon).to_dict()

ID1_map    = fe_key["ID1"].fillna("").astype(str).to_dict()

# tokenized for similarity

VendTok_map = fe_key["Vendor Name (clean)"].fillna("").astype(str).map(_tokens).to_dict()

DescTok_map = fe_key["Product Description (clean)"].fillna("").astype(str).map(_tokens).to_dict()

# ---------- (Optional) downsample pairs to label now ----------

pairs_to_label = pairs_cand

if MAX_ROWS_TO_LABEL is not None and len(pairs_to_label) > MAX_ROWS_TO_LABEL:

    pairs_to_label = pairs_to_label.sample(MAX_ROWS_TO_LABEL, random_state=17).reset_index(drop=True)

else:

    pairs_to_label = pairs_to_label.reset_index(drop=True)

print(f"Labeling {len(pairs_to_label):,} candidate pairs...")

# ---------- Label in chunks ----------

labeled_chunks = []

for start in range(0, len(pairs_to_label), CHUNK_SIZE):

    stop = min(start + CHUNK_SIZE, len(pairs_to_label))

    chunk = pairs_to_label.iloc[start:stop].copy()

    a = chunk["rec_id_A"].values

    b = chunk["rec_id_B"].values

    # exact/equality features

    vsn_eq   = np.fromiter((VSN_map.get(x,"") == VSN_map.get(y,"") for x,y in zip(a,b)), dtype=bool, count=len(chunk))

    dept_eq  = np.fromiter((Dept_map.get(x,"")  == Dept_map.get(y,"") for x,y in zip(a,b)), dtype=bool, count=len(chunk))

    class_eq = np.fromiter((Class_map.get(x,"") == Class_map.get(y,"") for x,y in zip(a,b)), dtype=bool, count=len(chunk))

    cat_eq   = np.fromiter((Cat_map.get(x,"")   == Cat_map.get(y,"") for x,y in zip(a,b)), dtype=bool, count=len(chunk))

    id1_eq   = np.fromiter((ID1_map.get(x,"")   == ID1_map.get(y,"") for x,y in zip(a,b)), dtype=bool, count=len(chunk))

    # token Jaccard similarities

    vend_sim = np.fromiter(

        (jaccard(VendTok_map.get(x,frozenset()), VendTok_map.get(y,frozenset())) for x,y in zip(a,b)),

        dtype=float, count=len(chunk)

    )

    desc_sim = np.fromiter(

        (jaccard(DescTok_map.get(x,frozenset()), DescTok_map.get(y,frozenset())) for x,y in zip(a,b)),

        dtype=float, count=len(chunk)

    )

    # attach features for inspection (optional but useful)

    chunk["vsn_equal"]   = vsn_eq

    chunk["dept_equal"]  = dept_eq

    chunk["class_equal"] = class_eq

    chunk["cat_equal"]   = cat_eq

    chunk["id1_equal"]   = id1_eq

    chunk["vendor_jacc"] = vend_sim

    chunk["desc_jacc"]   = desc_sim

    # ---------- Weak rules ----------

    # Positives (very conservative)

    pos_rules = (

        id1_eq |                                        # same deterministic umbrella -> strong positive

        (vsn_eq & (vend_sim >= 0.50)) |                 # same VSN and vendor reasonably close

        ((vend_sim >= VENDOR_POS_JACC) & (desc_sim >= DESC_POS_JACC))  # name + desc both high

    )

    # Negatives (very conservative)

    neg_rules = (

        ((vend_sim <= VENDOR_NEG_JACC) & (desc_sim <= DESC_NEG_JACC)) |  # both very low

        ((~dept_eq) & (~class_eq) & (~cat_eq) & (desc_sim <= 0.30))      # different hierarchy & low desc overlap

    )

    # Initialize label as -1 (unlabeled), then set positives/negatives

    chunk["label"] = -1

    chunk.loc[pos_rules, "label"] = 1

    chunk.loc[neg_rules, "label"] = 0

    labeled_chunks.append(chunk)

    print(f"  chunk {start:>9,}:{stop:>9,}  →  +{int(pos_rules.sum()):,} / -{int(neg_rules.sum()):,} labeled")

pairs_labeled = pd.concat(labeled_chunks, ignore_index=True)

# Keep only rows that received a label (weakly labeled set)

pairs_ws = pairs_labeled.loc[pairs_labeled["label"] >= 0].reset_index(drop=True)

print("\n Weak labeling complete.")

print(f"Total candidates considered: {len(pairs_to_label):,}")

print(f"Labeled positives: {int((pairs_ws['label']==1).sum()):,}")

print(f"Labeled negatives: {int((pairs_ws['label']==0).sum()):,}")

print(f"Unlabeled dropped: {len(pairs_to_label) - len(pairs_ws):,}")

# Optional: quick class balance + a few examples

print("\nClass balance:")

print(pairs_ws["label"].value_counts(dropna=False))

# Peek at some labeled pairs (joins back texts)

peek_cols = ["rec_id","Vendor Style (clean)","Vendor Name (clean)","Product Description (clean)"]

peek_ws = (

    pairs_ws

      .merge(df_fe[peek_cols], left_on="rec_id_A", right_on="rec_id", how="left")

      .rename(columns={

          "Vendor Style (clean)":"VSN_A",

          "Vendor Name (clean)":"Vendor_A",

          "Product Description (clean)":"Desc_A"

      }).drop(columns=["rec_id"])

      .merge(df_fe[peek_cols], left_on="rec_id_B", right_on="rec_id", how="left")

      .rename(columns={

          "Vendor Style (clean)":"VSN_B",

          "Vendor Name (clean)":"Vendor_B",

          "Product Description (clean)":"Desc_B"

      }).drop(columns=["rec_id"])

)

# Show a few strong positives and negatives for a sanity check

pos_examples = peek_ws.loc[peek_ws["label"]==1].sort_values(["id1_equal","vsn_equal","vendor_jacc","desc_jacc"], ascending=False).head(10)

neg_examples = peek_ws.loc[peek_ws["label"]==0].sort_values(["vendor_jacc","desc_jacc"], ascending=True).head(10)

print("\nTop 10 weak POS examples:")

print(pos_examples[["VSN_A","VSN_B","Vendor_A","Vendor_B","Desc_A","Desc_B","vendor_jacc","desc_jacc"]].to_string(index=False)[:2000])

print("\nTop 10 weak NEG examples:")

print(neg_examples[["VSN_A","VSN_B","Vendor_A","Vendor_B","Desc_A","Desc_B","vendor_jacc","desc_jacc"]].to_string(index=False)[:2000])
 

### Feature Vectorization

In [0]:
# ============================================

# Feature vectorization for pair classification

# expects: pairs_ws with columns like

#   rec_id_A, rec_id_B, label (0/1 on weak labels)

#   vsn_equal, vendor_jacc, desc_jacc,

#   dept_equal, class_equal, cat_equal,

#   score, signal

# ============================================

import numpy as np

import pandas as pd

def build_pair_features(pairs_ws: pd.DataFrame):

    df = pairs_ws.copy()

    # ---- 1) Make sure expected columns exist (fill if missing) ----

    needed = [

        "label", "vsn_equal", "vendor_jacc", "desc_jacc",

        "dept_equal", "class_equal", "cat_equal",

        "score", "signal"

    ]

    for c in needed:

        if c not in df.columns:

            df[c] = np.nan

    # ---- 2) Basic cleaning / coercion ----

    # Booleans → {0,1}

    for c in ["vsn_equal", "dept_equal", "class_equal", "cat_equal"]:

        df[c] = df[c].astype(float)  # True/False/NaN -> 1.0/0.0/NaN

        df[c] = df[c].fillna(0.0).clip(0.0, 1.0)

    # Similarities → float in [0,1]

    for c in ["vendor_jacc", "desc_jacc"]:

        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).clip(0.0, 1.0)

    # Score (from blocking) → bounded + log scaled variant

    df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0.0)

    df["score_clip"] = df["score"].clip(lower=0.0)

    df["score_log1p"] = np.log1p(df["score_clip"])

    # ---- 3) One-hot for blocking signal (categorical) ----

    # e.g., VENDOR_1-2, VSN_3-5, DESC_1-2

    sig = df["signal"].astype(str).fillna("UNK")

    sig_dummies = pd.get_dummies(sig, prefix="SIG", drop_first=False)

    # ---- 4) Simple interactions (often helpful) ----

    df["text_sim_mean"] = (df["vendor_jacc"] + df["desc_jacc"]) / 2.0

    df["text_sim_max"]  = np.maximum(df["vendor_jacc"], df["desc_jacc"])

    df["vsn_and_vendor"] = df["vsn_equal"] * df["vendor_jacc"]

    df["vsn_and_desc"]   = df["vsn_equal"] * df["desc_jacc"]

    df["hier_agree_cnt"] = df[["dept_equal", "class_equal", "cat_equal"]].sum(axis=1)

    # ---- 5) Assemble final feature frame ----

    base_feats = [

        "vsn_equal",

        "vendor_jacc", "desc_jacc",

        "text_sim_mean", "text_sim_max",

        "vsn_and_vendor", "vsn_and_desc",

        "dept_equal", "class_equal", "cat_equal",

        "hier_agree_cnt",

        "score_clip", "score_log1p",

    ]

    # keep only those that actually exist (defensive)

    base_feats = [c for c in base_feats if c in df.columns]

    df_features = pd.concat([df[base_feats], sig_dummies], axis=1)

    # Final safety: replace any residual NaNs with 0

    df_features = df_features.fillna(0.0)

    # ---- 6) Target vector (weak labels) ----

    y = pd.to_numeric(df["label"], errors="coerce").fillna(0).astype(int)

    # ---- 7) Outputs ----

    X = df_features.values.astype(np.float32)

    feature_cols = list(df_features.columns)

    print(" Feature vectorization complete")

    print(f"Pairs: {len(df_features):,}")

    print(f"Features: {len(feature_cols)}")

    print("Sample features:", feature_cols[:10])

    return X, y, feature_cols, df_features
 

In [0]:
# Call the function
X, y, feature_cols, df_feat = build_pair_features(pairs_ws)
# Quick sanity checks / peek
print("X shape:", X.shape)
print("y distribution:", pd.Series(y).value_counts(dropna=False).to_dict())
print("First 8 feature names:", feature_cols[:8])
df_feat.head(10)

In [0]:
print(pairs_ws.columns.tolist())

In [0]:
# ===== Train + calibrate Gradient Boosted Trees (with progress) =====

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import precision_recall_curve, f1_score

from sklearn.utils.class_weight import compute_sample_weight


assert 'X' in globals() and 'y' in globals(), "Run the feature vectorization cell first."

# ----- 1) stratified split -----

X_train, X_val, y_train, y_val = train_test_split(

    X, y, test_size=0.2, random_state=17, stratify=y

)

print(f"Train size: {X_train.shape[0]:,} | Val size: {X_val.shape[0]:,}")

print(f"Pos rate (train/val): {y_train.mean():.4f} / {y_val.mean():.4f}")

# ----- 2) model (no class_weight; use sample_weight instead) -----

gb = HistGradientBoostingClassifier(

    learning_rate=0.001,

    max_depth=None,          # let it grow; regularize via min_samples_leaf/L2

    max_iter=500,            # number of boosting iterations (trees)

    l2_regularization=0.0,

    min_samples_leaf=20,

    early_stopping=True,

    validation_fraction=0.1, # internal early-stopping split (from training set)

    n_iter_no_change=10,

    random_state=17,

    verbose=1                # <-- prints progress each iteration

)

# Handle class imbalance with per-sample weights

sw_train = compute_sample_weight(class_weight="balanced", y=y_train)

# Fit (progress prints during training)

gb.fit(X_train, y_train, sample_weight=sw_train)

# ----- 3) probability calibration on validation split -----

# Use sigmoid calibration; train calibrator on the held-out validation set

cal = CalibratedClassifierCV(gb, method="sigmoid", cv="prefit")

cal.fit(X_val, y_val)

# ----- 4) choose a probability threshold (maximize F1 on val) -----

probs_val = cal.predict_proba(X_val)[:, 1]

prec, rec, thr = precision_recall_curve(y_val, probs_val)

# Avoid division by zero

f1_vals = np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec), 0)

best_idx = int(np.argmax(f1_vals))

chosen_threshold = thr[best_idx] if best_idx < len(thr) else 0.5

print(f"\nChosen threshold (max F1): {chosen_threshold:.3f} | "

      f"Best F1: {f1_vals[best_idx]:.4f} | "

      f"Precision: {prec[best_idx]:.4f} | Recall: {rec[best_idx]:.4f}")

# Convenience scorer for any new pair-feature matrix

def score_pairs(pairs_feat_matrix, threshold=chosen_threshold, return_proba=True):

    """

    Score a pairs feature matrix with the calibrated model.

    Returns (proba, label) or just label depending on return_proba.

    """

    probs = cal.predict_proba(pairs_feat_matrix)[:, 1]

    labels = (probs >= threshold).astype(int)

    return (probs, labels) if return_proba else labels

print("\nModel trained, calibrated, and threshold selected.")



print("Use `score_pairs(X_new)` to score new candidate pairs.")
 

In [0]:


####HOLDOUT BLOCK, DONT DO UNTIL DOUBT OF OVERFITTING

from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.metrics import classification_report, roc_auc_score, make_scorer, f1_score

from sklearn.inspection import permutation_importance

import numpy as np

import pandas as pd

import time

# ---  Holdout evaluation ---

print("Running holdout validation check...")

t0 = time.time()

X_train2, X_holdout, y_train2, y_holdout = train_test_split(

    X, y, test_size=0.2, random_state=42, stratify=y

)

gb.fit(X_train2, y_train2)

preds = gb.predict(X_holdout)

probs = gb.predict_proba(X_holdout)[:, 1]

print("\n Holdout Evaluation Metrics:")

print(classification_report(y_holdout, preds, digits=4))

print("ROC-AUC:", round(roc_auc_score(y_holdout, probs), 6))

# ---  Feature importance via permutation ---

print("\nAnalyzing feature importance...")

perm = permutation_importance(gb, X_holdout, y_holdout, n_repeats=5, random_state=42)

imp = pd.Series(perm.importances_mean, index=feature_cols).sort_values(ascending=False)

print("\nTop 10 important features:")

print(imp.head(10))

top_contrib = 100 * np.sum(sorted(perm.importances_mean, reverse=True)[:2])

print(f"\nTop 2 features contribute ≈ {top_contrib:.2f}% of total importance")

# ---  Cross-validation check ---

print("\nRunning 5-fold cross-validation (F1 score)...")

cv_scores = cross_val_score(gb, X, y, cv=5, scoring=make_scorer(f1_score))

print("F1 across folds:", np.round(cv_scores, 4))

print(f"Mean F1 = {cv_scores.mean():.4f} | Std = {cv_scores.std():.4f}")

# ---  Simple overfitting heuristic ---

overfit_flag = False

if cv_scores.std() > 0.05 or cv_scores.mean() < 0.9:

    overfit_flag = True

if top_contrib > 80:

    overfit_flag = True

print("\n Overfitting diagnostic summary:")

print(f"→ Holdout F1: {f1_score(y_holdout, preds):.4f}")

print(f"→ CV mean/std: {cv_scores.mean():.4f} / {cv_scores.std():.4f}")

print(f"→ Dominant features (top2): {top_contrib:.2f}%")

if overfit_flag:

    print("\n Potential overfitting detected — model may be memorizing weak rules.")

    print("Consider: increase l2_regularization, add max_features=0.8, or simplify feature set.")

else:

    print("\n Model appears generalizable (no strong signs of overfitting).")

print(f"\nDone in {time.time() - t0:.2f}s")
 

In [0]:
# Step 1 — Rebuild df_feat to include rec_id_A and rec_id_B
# Ensure the number of rows match between features and candidate pairs
assert len(pairs_ws) == len(df_feat), \
   f"Row mismatch: pairs_ws({len(pairs_ws)}), df_feat({len(df_feat)})"
# Combine IDs with the feature matrix
df_feat = pd.concat([pairs_ws[["rec_id_A", "rec_id_B"]].reset_index(drop=True),
                    df_feat.reset_index(drop=True)], axis=1)
print(" Reattached rec_id_A and rec_id_B to df_feat")
print(df_feat.head(3))

In [0]:
# Step 2 — Sanity check before using in scoring or grouping
assert {'rec_id_A', 'rec_id_B'}.issubset(df_feat.columns), "IDs missing still!"
print("df_feat columns:", df_feat.columns.tolist())

In [0]:
# --- Align features while keeping rec_id columns for later grouping ---

#  Use the same feature list as during training

feature_cols = [

    'vsn_equal', 'vendor_jacc', 'desc_jacc', 'text_sim_mean', 'text_sim_max',

    'vsn_and_vendor', 'vsn_and_desc', 'dept_equal', 'class_equal', 'cat_equal',

    'hier_agree_cnt', 'score_clip', 'score_log1p', 'SIG_DESC_1-2',

    'SIG_VENDOR_1-2', 'SIG_VSN_3-5'

]

#  Check that these all exist in df_feat

missing = [c for c in feature_cols if c not in df_feat.columns]

if missing:

    print(f" Missing expected columns: {missing}")

else:

    print(f" All {len(feature_cols)} feature columns found")

#  Prepare X for scoring (only feature columns)

X_pairs = df_feat[feature_cols].copy()

#  Confirm dimensions

print(f"Model will score {X_pairs.shape[0]} pairs using {X_pairs.shape[1]} features")
 

In [0]:
from tqdm import tqdm

# --- SCORE ALL PAIRS ---

print("\nScoring candidate pairs...")

# Use your calibrated model

probs = cal.predict_proba(X_pairs)[:, 1]

# Add prediction scores and binary match labels

df_feat["score"] = probs

df_feat["match_label"] = (df_feat["score"] >= 0.5).astype(int)  # 0.5 = threshold, adjust if needed

# Summary

total_pairs = len(df_feat)

positive_pairs = int(df_feat["match_label"].sum())

print(f"Scoring complete — {positive_pairs:,} positive matches out of {total_pairs:,} total pairs")

print(f"Average predicted score: {df_feat['score'].mean():.4f}")

# Quick preview

print(df_feat[['rec_id_A', 'rec_id_B', 'score', 'match_label']].head(10))
 

In [0]:
from tqdm import tqdm

# --------------------------------------------

#  BUILD GROUPS FROM MATCHED PAIRS

# --------------------------------------------

print("\nBuilding ML-based groups...")

# Step 1:  Use only pairs predicted as matches

matched_pairs = df_feat.loc[df_feat["match_label"] == 1, ["rec_id_A", "rec_id_B"]].to_numpy()

print(f"Positive match pairs to cluster: {len(matched_pairs):,}")

# Step 2:  Union–Find data structure

class UnionFind:

    def __init__(self):

        self.parent = {}

    def find(self, x):

        if x not in self.parent:

            self.parent[x] = x

        if self.parent[x] != x:

            self.parent[x] = self.find(self.parent[x])

        return self.parent[x]

    def union(self, x, y):

        rx, ry = self.find(x), self.find(y)

        if rx != ry:

            self.parent[ry] = rx

# Step 3:  Merge all connected records

uf = UnionFind()

for a, b in tqdm(matched_pairs, desc="Merging groups"):

    uf.union(a, b)

# Step 4:  Assign group IDs

print("\nExtracting group assignments...")

root_to_gid = {}

gid_counter = 0

rec_to_gid = {}

for rec_id in pd.unique(df_feat[["rec_id_A", "rec_id_B"]].values.ravel()):

    root = uf.find(rec_id)

    if root not in root_to_gid:

        gid_counter += 1

        root_to_gid[root] = gid_counter

    rec_to_gid[rec_id] = root_to_gid[root]

# Step 5:  Attach group IDs back to a DataFrame

df_groups = pd.DataFrame(list(rec_to_gid.items()), columns=["rec_id", "ID_ML"])

print(f" Grouping complete — {gid_counter:,} unique ML groups created.")

print(df_groups.head(10))
 

In [0]:
# ============================================================

# BLOCK 4: REPORTING AND SANITY CHECKS (corrected)

# ============================================================

print("\n===== FINAL GROUPING SUMMARY =====")

# Ensure we start from df_groups (the ML grouping result)

df_groups_final = df_groups.copy()

# -----------------------------------------

# STEP 1: Join back original deterministic ID (ID1)

# -----------------------------------------

if "ID1" in df_fe.columns and "rec_id" in df_groups_final.columns:

    df_groups_final = df_groups_final.merge(

        df_fe[["rec_id", "ID1"]],

        on="rec_id",

        how="left"

    )

else:

    print(" Either df_fe.ID1 or rec_id not found — skipping join for ID1.")

# -----------------------------------------

# STEP 2: Summary counts

# -----------------------------------------

n_records = len(df_groups_final)

n_new_groups = df_groups_final["ID_ML"].nunique(dropna=True)

print(f" Records processed: {n_records:,}")

print(f" Unique ML-based groups formed: {n_new_groups:,}")

if "ID1" in df_groups_final.columns:

    n_old_groups = df_groups_final["ID1"].nunique()

    print(f" Original deterministic groups (ID1): {n_old_groups:,}")

    print(f" Change ratio: {(100 * (1 - n_new_groups / n_old_groups)):.2f}% fewer groups after ML merging")

else:

    print(" Column 'ID1' missing even after join — skipping comparison.")

# -----------------------------------------

# STEP 3: Group size stats

# -----------------------------------------

group_sizes = df_groups_final.groupby("ID_ML").size()

print("\n===== GROUP SIZE DISTRIBUTION =====")

print(group_sizes.describe())

# -----------------------------------------

# STEP 4: Identify merges (ML groups containing multiple old IDs)

# -----------------------------------------

if "ID1" in df_groups_final.columns:

    merged_cases = (

        df_groups_final.groupby("ID_ML")["ID1"]

        .nunique()

        .reset_index(name="unique_old_ids")

        .query("unique_old_ids > 1")

    )

    print(f"\n ML groups that merged multiple old IDs: {len(merged_cases):,}")

    if len(merged_cases) > 0:

        print("Sample merged groups:")

        print(merged_cases.head(10).to_string(index=False))

else:

    print("\n(No ID1 column available to check merges.)")

# -----------------------------------------

# STEP 5: Keep master copy for downstream export

# -----------------------------------------

df_groups_final = df_groups_final.copy()

print("\n Block 4 complete — reporting finished.")
 

In [0]:
df_all.columns

In [0]:
###newwwwww
# ============================================================

# BLOCK 5: BEFORE vs AFTER GROUPING — FULL DATA COMPARISON VIEW

# ============================================================

# --- Step 1: Merge original deterministic ID1 with new ML grouping ---

if "ID1" not in df_groups_final.columns and "rec_id" in df_groups_final.columns:

    # In case ID1 wasn’t already joined earlier

    df_groups_final = df_groups_final.merge(

        df_fe[["rec_id", "ID1"]],

        on="rec_id",

        how="left"

    )

# --- Step 2: Merge back with ALL original columns for complete view ---

# This ensures you have totals, POs, SKUs, etc. alongside the IDs

df_compare = df_top_100k.merge(
   df_groups_final[["rec_id", "ID1", "ID_ML"]],
   on="rec_id",
   how="left",
   validate="m:1"
)

# --- Optional Step 6: Save or view ---

# display(df_compare.head(50))

# df_compare.to_parquet("/dbfs/tmp/compare_ID1_vs_IDML_full.parquet", index=False)

print("\nBlock 5 complete — full comparison dataframe ready (df_compare).")
 

In [0]:
df_compare.columns

In [0]:
# ============================================================

# BLOCK 5: BEFORE vs AFTER GROUPING COMPARISON VIEW

# ============================================================

# --- Step 1: Merge original deterministic ID1 with new ML grouping ---

if "ID1" not in df_groups_final.columns and "rec_id" in df_groups_final.columns:

    # Just in case you didn’t already merge ID1 earlier

    df_groups_final = df_groups_final.merge(

        df_fe[["rec_id", "ID1"]],

        on="rec_id",

        how="left"

    )

# --- Step 2: Keep clean comparison view ---

cols_to_keep = ["rec_id", "ID1", "ID_ML", 

                "Vendor Name (clean)", "Vendor Style (clean)", 

                "Product Description (clean)"]

df_compare = df_groups_final.loc[:, [c for c in cols_to_keep if c in df_groups_final.columns]].copy()

# --- Step 3: Add merge indicator flags ---

df_compare["same_group"] = (df_compare["ID1"] == df_compare["ID_ML"])

df_compare["group_changed"] = ~df_compare["same_group"]

# --- Step 4: Print summary stats ---

changed = df_compare["group_changed"].sum()

print(f"\n Records that moved to a different ML group: {changed:,} of {len(df_compare):,} "

      f"({changed/len(df_compare)*100:.2f}%)")

print("\n===== Sample of Changed Group Assignments =====")

sample_diff = df_compare.query("group_changed").head(10)

print(sample_diff.to_string(index=False))

# --- Optional Step 5: Save or display ---

# display(df_compare.head(50))  

# df_compare.to_csv("compare_ID1_vs_IDML.csv", index=False)

print("\n Block 5 complete — comparison dataframe ready (df_compare).")
 

In [0]:
# ============================================================

# BLOCK — MERGE ORIGINAL DATA (with POs etc.) WITH NEW ML GROUP IDS

# ============================================================

# Make sure both have rec_id

assert "rec_id" in df_all.columns, "df_all must contain rec_id"

assert "rec_id" in df_groups.columns, "df_groups must contain rec_id"

# Keep only essential grouping columns

df_groups_slim = df_groups[["rec_id", "ID_ML"]].copy()

# Merge ML-based groups into the full detailed dataset

df_combined = df_all.merge(df_groups_slim, on="rec_id", how="left", validate="m:1")

print(f"Combined dataframe created: {len(df_combined):,} records")

print("Columns now include transactional fields + ID1 + ID_ML.")

# Optional preview

cols_preview = [c for c in ["rec_id", "ID1", "ID_ML", "PO", "PO_pre", "SKU", "Total Units Ordered"] if c in df_combined.columns]

print(df_combined[cols_preview].head(10))

# df_combined now holds your full enriched dataset ready for export
 

In [0]:
df_combined.columns

In [0]:
###============================================================
# BLOCK — DROP RECORDS WITHOUT ML GROUP ASSIGNMENT
# ============================================================
before = len(df_compare)
df_compare = df_compare.dropna(subset=["ID_ML"]).copy()
after = len(df_compare)
print(f" Dropped {before - after:,} records with missing ID_ML.")
print(f"Remaining records in df_combined: {after:,}")

In [0]:
df_compare.to_csv("/dbfs/FileStore/FinalCombinedNewLatest.csv", index=False, encoding = "utf-8-sig")


# Getting the workspace URL

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")  


# File path relative to /dbfs/FileStore

file_name = "FinalCombinedNewLatest.csv"

download_url = f"https://{workspace_url}/files/{file_name}"

print("Download link:", download_url)
 

In [0]:
# ============================================================

# BLOCK — FIND CROSS-REGION ML GROUPS

# ============================================================

# Step 1: Identify ML groups with >1 unique source_region

cross_region_groups = (

    df_combined.groupby("ID_ML")["source_region"]

    .nunique()

    .reset_index()

    .query("source_region > 1")["ID_ML"]

    .tolist()

)

# Step 2: Filter main dataframe for those groups

df_cross_region = df_combined[df_combined["ID_ML"].isin(cross_region_groups)].copy()

# Step 3: Print quick summary

print(f" Cross-region ML groups found: {len(cross_region_groups):,}")

print(f"Total records in these groups: {len(df_cross_region):,}")

# Optional: quick preview

cols_preview = [c for c in ["ID_ML", "source_region", "Vendor Name (clean)", "Vendor Style (clean)", "Product Description (clean)"] if c in df_cross_region.columns]

display(df_cross_region[cols_preview].head(20))
 

In [0]:
df_cross_region.to_csv("/dbfs/FileStore/CrossRegionMatches.csv", index=False, encoding = "utf-8-sig")


# Getting the workspace URL

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")  


# File path relative to /dbfs/FileStore

file_name = "CrossRegionMatches.csv"

download_url = f"https://{workspace_url}/files/{file_name}"

print("Download link:", download_url)
 


### Prepare `df_compare` for Rule Enforcement — Input Checks & Text Normalization

**What this cell does**  
- Verifies that the **model output frame** `df_compare` is present and contains the columns needed for rule enforcement (e.g., `PS_PRODUCT_ID`, `vendor_name`, `vendor_style`, `class_name`, `category_name`, and a stable row key like `rec_id`).  
- Creates **normalized** versions of the vendor, style, class, and category fields (lowercasing, trimming, punctuation/diacritics cleanup, optional alias mapping) to eliminate accidental mismatches before applying hard rules.  
- Logs shape, missing‑value counts, and a small sample to confirm readiness.

**Why it matters downstream (business)**  
- Ensures that **hard domain rules** (e.g., *same vendor name + same vendor style*, or *same vendor name + same/similar class/category*) are evaluated on **consistent text** so that legitimate equivalences are not missed due to casing, spacing, or minor spelling differences.  
- Reduces false fragmentation so final product groupings in **Power BI** reflect how Product Compliance and Merchandising actually identify products across banners and regions. This is the foundation for the **cascading group unions** that produce `PS_PRODUCT_ID_NEW`. [1](https://tjxinc-my.sharepoint.com/personal/vig00035_tjx_com/Documents/Microsoft%20Copilot%20Chat%20Files/Product%20Matcher%20Clone.html)

**How it works (technical)**  
- **Input:** `df_compare` (produced by your scoring/grouping steps prior to export in Cell ~75). Expected columns:  
  - IDs: `PS_PRODUCT_ID` (model group id), `rec_id` (stable row key)  
  - Rule fields: `vendor_name`, `vendor_style`, `class_name`, `category_name`  
- **Normalize:** create `vendor_name_norm`, `vendor_style_norm`, `class_name_norm`, `category_name_norm` using:  
  - lowercase; trim; collapse internal whitespace; strip punctuation/diacritics; optional alias/lookup mapping for known vendor variants.  
- **Traceability:** emit row counts, null counts for the rule columns (raw & normalized), and `display()` a 10‑row sample so both technical and non‑technical reviewers can validate.  
- **Output for next step:** a clean `df_compare_norm` (or updated `df_compare`) with the normalized columns, ready for building **rule keys** and **PS‑group edges** in the next block. [1](https://tjxinc-my.sharepoint.com/personal/vig00035_tjx_com/Documents/Microsoft%20Copilot%20Chat%20Files/Product%20Matcher%20Clone.html)


In [0]:
df_compare = spark.read.csv(
   "dbfs:/mnt/analytics/riskcomp_corp/output/Product_Matcher/FinalCombinedNewLatest.csv",
   header=True,
   inferSchema=True
)

In [0]:

# STEP 1 — Input checks & text normalization for df_compare (Spark DataFrame)

from pyspark.sql import functions as F, types as T
import re, unicodedata

# ---------- 1) Sanity checks: presence of required columns ----------
required_cols = [
    "rec_id", "ID_ML",               # keys
    "Vendor Name (clean)", "Vendor Style (clean)",  # rule fields (exact)
    "Class Name", "Category Name", "Dept"           # taxonomy rule fields
]

missing = [c for c in required_cols if c not in df_compare.columns]
if missing:
    raise ValueError(f"[Step 1] Missing required columns in df_compare: {missing}")

# ---------- 2) Create a canonical deterministic ID1 (prefer ID1_y then ID1_x) ----------
df_compare_norm = df_compare.withColumn(
    "ID1",
    F.coalesce(F.col("ID1_y"), F.col("ID1_x"))
)

# ---------- 3) Normalization helpers (Spark UDFs) ----------
_punct_re = re.compile(r"[^\w\s]+", flags=re.UNICODE)

def _strip_diacritics_py(s: str) -> str:
    if s is None:
        return None
    nfkd = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in nfkd if not unicodedata.combining(ch))

def _normalize_text_py(x):
    if x is None:
        return None
    s = str(x)
    s = _strip_diacritics_py(s)
    s = s.lower()
    s = _punct_re.sub(" ", s)        # replace punctuation with space
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

normalize_text_udf = F.udf(_normalize_text_py, T.StringType())

# Optional vendor aliasing (leave empty unless you have known canonicalizations)
VENDOR_ALIAS_MAP = {
    # "tjx co": "tjx",
    # "tjx companies": "tjx",
}
bc_vendor_alias = spark.sparkContext.broadcast(VENDOR_ALIAS_MAP)

@F.udf(returnType=T.StringType())
def vendor_alias_udf(s):
    if s is None:
        return None
    m = bc_vendor_alias.value
    return m.get(s, s)

# ---------- 4) Build normalized columns used by hard rules ----------
df_compare_norm = (
    df_compare_norm
        .withColumn("vendor_name_norm_base",  normalize_text_udf(F.col("Vendor Name (clean)")))
        .withColumn("vendor_name_norm",       vendor_alias_udf(F.col("vendor_name_norm_base")))
        .withColumn("vendor_style_norm",      normalize_text_udf(F.col("Vendor Style (clean)")))
        .withColumn("class_name_norm",        normalize_text_udf(F.col("Class Name")))
        .withColumn("category_name_norm",     normalize_text_udf(F.col("Category Name")))
        .withColumn("dept_norm",              normalize_text_udf(F.col("Dept")))
)

# ---------- 5) Basic diagnostics ----------
print("\n[Step 1] df_compare_norm ready for rule enforcement.")

rows = df_compare_norm.count()
print(f"Rows: {rows:,}")

# Null counts for key & rule fields
cols_to_check = [
    "rec_id", "ID_ML", "ID1",
    "Vendor Name (clean)", "Vendor Style (clean)", "Class Name", "Category Name", "Dept",
    "vendor_name_norm", "vendor_style_norm", "class_name_norm", "category_name_norm", "dept_norm"
]

agg_exprs = [
    F.sum(F.when(F.col(c).isNull() | (F.col(c) == ""), 1).otherwise(0)).alias(c)
    for c in cols_to_check
]
null_counts = df_compare_norm.agg(*agg_exprs).toPandas().T
null_counts.columns = ["null_or_empty_count"]
print("\nNull/empty counts (key & rule fields):")
print(null_counts.sort_values("null_or_empty_count", ascending=False).to_string())

# Small preview for business & technical review
preview_cols = [
    "rec_id", "ID1", "ID_ML",
    "Vendor Name (clean)", "Vendor Style (clean)", "Class Name", "Category Name", "Dept",
    "vendor_name_norm", "vendor_style_norm", "class_name_norm", "category_name_norm", "dept_norm"
]
display(df_compare_norm.select([c for c in preview_cols if c in df_compare_norm.columns]).limit(10))

# ---------- 6) Hand-off note ----------
# Next step (Block 2) will:
# - Construct rule keys (e.g., vendor_name_norm + vendor_style_norm; vendor_name_norm + class/category with similarity)
# - Build PS-group-level edges (between ID_ML groups) wherever a rule triggers
# - Compute connected components to assign PS_PRODUCT_ID_NEW (final group id)
# - Append lineage: RULE_KEYS_HIT, RULESET_VERSION



### Construct Rule Keys & PS‑Group Edges — Set Up for Cascading Unions

**What this cell does**  
- Builds **rule keys** that capture your hard “must‑merge” logic:  
  1) **R1_VENDOR_STYLE (exact):** same `vendor_name_norm` **and** same `vendor_style_norm` ⇒ link their ML groups.  
  2) **R2_VENDOR_CLASS (exact or similar):** same `vendor_name_norm` **and** same/very‑similar `class_name_norm` ⇒ link their ML groups.  
  3) **R3_VENDOR_CATEGORY (exact or similar):** same `vendor_name_norm` **and** same/very‑similar `category_name_norm` ⇒ link their ML groups.  
- For each rule that hits across **different** `ID_ML` values, creates **edges between PS groups** (nodes = `ID_ML`). These edges mean “place these groups in the **same final component**.”  
- Produces a compact **edge list** (e.g., `src_pid`, `dst_pid`, `rule_family`) that we’ll feed into Step 3 to compute **connected components** and derive `PS_PRODUCT_ID_NEW`.

**Why it matters downstream (business)**  
- Encodes Product Compliance’s directives so that if **any item** in PS group **A** and **any item** in PS group **B** share the rule key, then **all items** in **A ∪ B** end up in a single final group. This supports the transitive, group‑level behavior you asked for and ensures the output for **Power BI** reflects real‑world product identity, not just model similarity. The notebook you shared already culminates in a FileStore export; we’re inserting this logic just before that export. [1](https://tjxinc-my.sharepoint.com/personal/vig00035_tjx_com/Documents/Microsoft%20Copilot%20Chat%20Files/Product%20Matcher%20Clone.html)

**How it works (technical)**  
- **Input:** `df_compare_norm` from Step 1 (Spark DF) with:  
  - Keys: `rec_id`, `ID_ML`, `ID1`  
  - Normalized fields: `vendor_name_norm`, `vendor_style_norm`, `class_name_norm`, `category_name_norm`, `dept_norm`  
- **Rule keys & similarity gates:**  
  - **R1 (exact):** `(vendor_name_norm, vendor_style_norm)` must be **non‑null** and equal.  
  - **R2/R3 (exact or similar):** For similarity on class/category, we’ll create normalized tokens and use a **high threshold** (e.g., token‑Jaccard or RapidFuzz if needed later). For this step’s edges we’ll stage both **exact** and **similar** variants and keep thresholds **conservative** to avoid over‑merging.  
- **From rows to groups:**  
  - Within each rule family, group by the **rule key** and collect the set of **distinct `ID_ML`** values.  
  - Whenever a rule key maps to **>1** distinct `ID_ML`, emit **edges** connecting those `ID_ML` values (we’ll star‑anchor to the smallest ID to avoid O(K²) pairs).  
  - Tag each edge with `rule_family` so we can audit why a merge occurred.  
- **Output for next step:** a **de‑duplicated edge list** (undirected) connecting `ID_ML` groups that must be unified, ready for **connected components** in Step 3.  
- **Placement in notebook:** This sits **after** your compare‑frame build and **before** the current FileStore export (which in your notebook happens around Cell 75 writing `FinalCombinedNewLatest.csv`). We will later swap that export to include `PS_PRODUCT_ID_NEW`. [1](https://tjxinc-my.sharepoint.com/personal/vig00035_tjx_com/Documents/Microsoft%20Copilot%20Chat%20Files/Product%20Matcher%20Clone.html)

**Notes & guardrails**  
- Keep a **conservative similarity threshold** for class/category to prevent accidental group explosions; anything borderline can be logged for review.  
- Log the **component sizes** after unions; flag any mega‑components for QA.  
- Preserve `RULESET_VERSION` and `rule_family` on edges for explainability when we join back results in the final step.  



### Step 2 — Extend PS‑Group Edges with New AND‑based Rules (R4–R7)

**What this cell does**  
- Adds four new **must‑merge** rules to your existing Step 2, each using **AND** logic for the “similar” parts:
  1) **R4_VENDOR_VSN_DESC_SIM**: same vendor **AND** (VSN similar **AND** Description similar).  
  2) **R5_VSN_DESC_SIM**: same VSN **AND** Description similar.  
  3) **R6_VENDOR_CLASS_CAT_DESC_SIM**: same vendor + class + category **AND** Description similar.  
  4) **R7_VENDOR_DESC_CLASS_SIM**: Vendor similar **AND** Description similar **AND** Class similar (optionally same dept to reduce pair count).

- Emits one consolidated `edges_rules` (adds R4–R7 to your existing R1–R3 edges).  
- Your **Step 3** (Union‑Find) then merges groups transitively to produce `PS_PRODUCT_ID_NEW`.

**AND semantics**  
- For each rule above, **all** “similar” checks in that rule must pass (e.g., R4 requires **VSN_SIM ≥ threshold AND DESC_SIM ≥ threshold**, under the same vendor).

**Guardrails**  
- Tokenize text with punctuation stripping and whitespace normalization; require **min token counts** to avoid spurious matches.  
- **R5** can span vendors if a VSN is reused; you can flip a switch (`R5_REQUIRE_SAME_VENDOR`) to demand same vendor as well.  
- **R7** is highest risk (no exact anchors); you can optionally require **same dept** (`R7_REQUIRE_SAME_DEPT=True`) or raise thresholds.  
- Degree/size diagnostics remain in Step 3; keep an eye on mega‑components.


In [0]:

# STEP 2 (extended, FIXED + compact VSN, R4 relaxed) — Build PS-group edges
# This cell fully replaces Step 2.
# Changes vs. prior version:
#   - Adds vendor_style_compact (alphanumeric-only VSN) for exact VSN checks (R1, R5).
#   - R4 is RELAXED: NO vendor-name requirement. Edge if (VSN_sim >= thr AND DESC_sim >= thr).
#   - Keeps your AND-gated R5–R7 and exact/sim R2–R3.

from pyspark.sql import functions as F, types as T

# ----------------------------
# Thresholds & switches
# ----------------------------
# Existing similarity gates (exact rules are separate)
CLASS_SIM_JACCARD     = 0.40
CATEGORY_SIM_JACCARD  = 0.40

# R4 (RELAXED) — VSN similar AND Description similar (NO vendor-name anchor)
VSN_SIM_MIN_R4        = 0.30
DESC_SIM_MIN_R4       = 0.30
R4_REQUIRE_SAME_DEPT  = False   # optional guard: compare only within same dept

# R5 — same VSN (COMPACT) AND Description similar (optionally same vendor)
DESC_SIM_MIN_R5       = 0.35
R5_REQUIRE_SAME_VENDOR= False

# R6 — same vendor + class + category AND Description similar
DESC_SIM_MIN_R6       = 0.35

# R7 — Vendor similar AND Description similar AND Class similar (optional: same dept)
VENDOR_SIM_MIN_R7     = 0.40
DESC_SIM_MIN_R7       = 0.40
CLASS_SIM_MIN_R7      = 0.40
R7_REQUIRE_SAME_DEPT  = False

# Minimum token counts to avoid spurious matches
MIN_TOKENS_VENDOR = 1
MIN_TOKENS_VSN    = 1
MIN_TOKENS_CLASS  = 1
MIN_TOKENS_DESC   = 2

# ----------------------------
# Helpers
# ----------------------------
def tokens_expr(colname):
    # distinct, non-empty word tokens
    return F.expr(
        f"filter(array_distinct(split(regexp_replace(coalesce({colname},''), '[^\\w]+', ' '), '\\\\s+')), x -> x <> '')"
    )

def jaccard_expr(a, b):
    # token-count guards ensure union > 0
    return (F.size(F.array_intersect(a, b)) / F.size(F.array_union(a, b)))

def star_edges_from_pids(df_groups, pids_col: str, rule_family: str):
    return (
        df_groups
          .withColumn("pids_sorted", F.array_sort(F.col(pids_col)))
          .withColumn("anchor_pid", F.element_at(F.col("pids_sorted"), 1))
          .withColumn("others", F.slice(F.col("pids_sorted"), 2, 1000000))
          .select("anchor_pid", F.explode_outer("others").alias("other_pid"))
          .filter(F.col("anchor_pid").isNotNull() & F.col("other_pid").isNotNull() & (F.col("anchor_pid") != F.col("other_pid")))
          .withColumn("src_pid", F.least(F.col("anchor_pid"), F.col("other_pid")))
          .withColumn("dst_pid", F.greatest(F.col("anchor_pid"), F.col("other_pid")))
          .select("src_pid","dst_pid").dropDuplicates()
          .withColumn("rule_family", F.lit(rule_family))
    )

# ----------------------------
# Preconditions & normalization
# ----------------------------
need = ["ID_ML","vendor_name_norm","vendor_style_norm","class_name_norm","category_name_norm"]
missing = [c for c in need if c not in df_compare_norm.columns]
if missing:
    raise ValueError(f"[Step 2] df_compare_norm missing columns: {missing}")

# Add desc_norm if missing
if "desc_norm" not in df_compare_norm.columns:
    df_compare_norm = df_compare_norm.withColumn(
        "desc_norm",
        F.trim(F.lower(F.regexp_replace(F.coalesce(F.col("Product Description (clean)"), F.lit("")), r"[^\w]+", " ")))
    )

# Add vendor_style_compact (lowercase alphanumeric-only) for exact VSN equality
df_compare_norm = df_compare_norm.withColumn(
    "vendor_style_compact",
    F.lower(F.regexp_replace(F.coalesce(F.col("vendor_style_norm"), F.lit("")), r"[^A-Za-z0-9]+", ""))
)

# Cast ID to string for graph nodes
df_base = df_compare_norm.withColumn("ID_ML_str", F.col("ID_ML").cast("string"))

# ----------------------------
# Build tokens & deduplicate per ID_ML
# ----------------------------
df_tok = (
    df_base
      .withColumn("vendor_tokens", tokens_expr("vendor_name_norm"))
      .withColumn("vsn_tokens",    tokens_expr("vendor_style_norm"))   # similarity uses token form
      .withColumn("class_tokens",  tokens_expr("class_name_norm"))
      .withColumn("cat_tokens",    tokens_expr("category_name_norm"))
      .withColumn("desc_tokens",   tokens_expr("desc_norm"))
      .withColumn("dept_key",      F.coalesce(F.col("dept_norm"), F.lit("__NO_DEPT__")))
)

cols_keep = ["ID_ML_str","vendor_name_norm","vendor_style_norm","vendor_style_compact",
             "class_name_norm","category_name_norm","dept_key",
             "vendor_tokens","vsn_tokens","class_tokens","cat_tokens","desc_tokens"]
df_g = df_tok.select(*cols_keep).dropDuplicates(["ID_ML_str"])

# ============================================================================
# R1 — EXACT: same vendor name AND same vendor style (using compact VSN)
# ============================================================================
r1_candidates = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & (F.col("vendor_style_compact") != ""))
      .select("vendor_name_norm","vendor_style_compact","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_name_norm","vendor_style_compact")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("pids") > 1)
)
edges_r1 = star_edges_from_pids(r1_candidates, "pids", "R1_VENDOR_STYLE")

# ============================================================================
# R2 — vendor + class_name: exact & similar
# ============================================================================
# Exact
r2a_candidates = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & F.col("class_name_norm").isNotNull())
      .select("vendor_name_norm","class_name_norm","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_name_norm","class_name_norm")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("pids") > 1)
)
edges_r2_exact = star_edges_from_pids(r2a_candidates, "pids", "R2_VENDOR_CLASS_EXACT")

# Similar
r2b_keys = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & F.col("class_name_norm").isNotNull())
      .select("vendor_name_norm","class_name_norm","class_tokens","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_name_norm","class_name_norm","class_tokens")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("class_tokens") > 0)
)
r2b_pairs = (
    r2b_keys.alias("a")
      .join(r2b_keys.alias("b"),
            (F.col("a.vendor_name_norm")==F.col("b.vendor_name_norm")) &
            (F.col("a.class_name_norm") < F.col("b.class_name_norm")),
            "inner")
      .withColumn("jacc", jaccard_expr(F.col("a.class_tokens"), F.col("b.class_tokens")))
      .filter(F.col("jacc") >= F.lit(CLASS_SIM_JACCARD))
      .select(F.array_distinct(F.array_union(F.col("a.pids"), F.col("b.pids"))).alias("pids_union"))
      .filter(F.size("pids_union") > 1)
)
edges_r2_sim = star_edges_from_pids(r2b_pairs.withColumnRenamed("pids_union","pids_union"), "pids_union", "R2_VENDOR_CLASS_SIM")

# ============================================================================
# R3 — vendor + category_name: exact & similar
# ============================================================================
# Exact
r3a_candidates = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & F.col("category_name_norm").isNotNull())
      .select("vendor_name_norm","category_name_norm","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_name_norm","category_name_norm")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("pids") > 1)
)
edges_r3_exact = star_edges_from_pids(r3a_candidates, "pids", "R3_VENDOR_CATEGORY_EXACT")

# Similar
r3b_keys = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & F.col("category_name_norm").isNotNull())
      .select("vendor_name_norm","category_name_norm","cat_tokens","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_name_norm","category_name_norm","cat_tokens")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("cat_tokens") > 0)
)
r3b_pairs = (
    r3b_keys.alias("a")
      .join(r3b_keys.alias("b"),
            (F.col("a.vendor_name_norm")==F.col("b.vendor_name_norm")) &
            (F.col("a.category_name_norm") < F.col("b.category_name_norm")),
            "inner")
      .withColumn("jacc", jaccard_expr(F.col("a.cat_tokens"), F.col("b.cat_tokens")))
      .filter(F.col("jacc") >= F.lit(CATEGORY_SIM_JACCARD))
      .select(F.array_distinct(F.array_union(F.col("a.pids"), F.col("b.pids"))).alias("pids_union"))
      .filter(F.size("pids_union") > 1)
)
edges_r3_sim = star_edges_from_pids(r3b_pairs.withColumnRenamed("pids_union","pids_union"), "pids_union", "R3_VENDOR_CATEGORY_SIM")

# ============================================================================
# R4 (RELAXED) — VSN similar AND Desc similar → edge  (NO vendor-name requirement)
# ============================================================================
r4_left  = df_g.select("ID_ML_str","dept_key","vsn_tokens","desc_tokens")
r4_right = (r4_left
            .withColumnRenamed("ID_ML_str","ID_ML_str_r")
            .withColumnRenamed("dept_key","dept_key_r")
            .withColumnRenamed("vsn_tokens","vsn_tokens_r")
            .withColumnRenamed("desc_tokens","desc_tokens_r"))

r4_dept_cond = ((F.col("a.dept_key")==F.col("b.dept_key_r"))
                if R4_REQUIRE_SAME_DEPT else F.lit(True))

r4_pairs = (r4_left.alias("a")
    .join(r4_right.alias("b"),
          (F.col("a.ID_ML_str") < F.col("b.ID_ML_str_r")) & r4_dept_cond,
          "inner")
    .withColumn("vsn_jacc",  jaccard_expr(F.col("a.vsn_tokens"),  F.col("b.vsn_tokens_r")))
    .withColumn("desc_jacc", jaccard_expr(F.col("a.desc_tokens"), F.col("b.desc_tokens_r")))
    .withColumn("ok_sizes",
        (F.size("a.vsn_tokens")  >= F.lit(MIN_TOKENS_VSN))  &
        (F.size("b.vsn_tokens_r")>= F.lit(MIN_TOKENS_VSN))  &
        (F.size("a.desc_tokens") >= F.lit(MIN_TOKENS_DESC)) &
        (F.size("b.desc_tokens_r")>=F.lit(MIN_TOKENS_DESC))
    )
    # AND semantics: both similarities must pass
    .filter(F.col("ok_sizes") &
            (F.col("vsn_jacc")  >= F.lit(VSN_SIM_MIN_R4)) &
            (F.col("desc_jacc") >= F.lit(DESC_SIM_MIN_R4)))
    .select(F.col("a.ID_ML_str").alias("src_pid"), F.col("b.ID_ML_str_r").alias("dst_pid"))
    .dropDuplicates()
    .withColumn("rule_family", F.lit("R4_VSN_DESC_SIM_NO_VENDOR"))
)

# ============================================================================
# R5 — same VSN (COMPACT) AND Desc similar (optional same vendor)
# ============================================================================
r5_left  = df_g.select("vendor_style_compact","vendor_name_norm","ID_ML_str","desc_tokens")
r5_right = (r5_left
            .withColumnRenamed("ID_ML_str","ID_ML_str_r")
            .withColumnRenamed("vendor_name_norm","vendor_name_norm_r")
            .withColumnRenamed("desc_tokens","desc_tokens_r"))

r5_vendor_cond = ((F.col("a.vendor_name_norm")==F.col("b.vendor_name_norm_r"))
                  if R5_REQUIRE_SAME_VENDOR else F.lit(True))

r5_pairs = (r5_left.alias("a")
    .join(r5_right.alias("b"),
          (F.col("a.vendor_style_compact") == F.col("b.vendor_style_compact")) &
          r5_vendor_cond &
          (F.col("a.ID_ML_str") < F.col("b.ID_ML_str_r")),
          "inner")
    .withColumn("desc_jacc", jaccard_expr(F.col("a.desc_tokens"), F.col("b.desc_tokens_r")))
    .withColumn("ok_sizes",
        (F.size("a.desc_tokens") >= F.lit(MIN_TOKENS_DESC)) &
        (F.size("b.desc_tokens_r")>=F.lit(MIN_TOKENS_DESC))
    )
    .filter(F.col("ok_sizes") & (F.col("desc_jacc") >= F.lit(DESC_SIM_MIN_R5)))
    .select(F.col("a.ID_ML_str").alias("src_pid"), F.col("b.ID_ML_str_r").alias("dst_pid"))
    .dropDuplicates()
    .withColumn("rule_family", F.lit("R5_VSN_DESC_SIM"))
)

# ============================================================================
# R6 — same vendor + class + category AND Desc similar
# ============================================================================
r6_left  = df_g.select("vendor_name_norm","class_name_norm","category_name_norm","ID_ML_str","desc_tokens")
r6_right = (r6_left
            .withColumnRenamed("ID_ML_str","ID_ML_str_r")
            .withColumnRenamed("desc_tokens","desc_tokens_r"))

r6_pairs = (r6_left.alias("a")
    .join(r6_right.alias("b"),
          (F.col("a.vendor_name_norm")==F.col("b.vendor_name_norm")) &
          (F.col("a.class_name_norm")==F.col("b.class_name_norm")) &
          (F.col("a.category_name_norm")==F.col("b.category_name_norm")) &
          (F.col("a.ID_ML_str") < F.col("b.ID_ML_str_r")),
          "inner")
    .withColumn("desc_jacc", jaccard_expr(F.col("a.desc_tokens"), F.col("b.desc_tokens_r")))
    .withColumn("ok_sizes",
        (F.size("a.desc_tokens") >= F.lit(MIN_TOKENS_DESC)) &
        (F.size("b.desc_tokens_r")>=F.lit(MIN_TOKENS_DESC))
    )
    .filter(F.col("ok_sizes") & (F.col("desc_jacc") >= F.lit(DESC_SIM_MIN_R6)))
    .select(F.col("a.ID_ML_str").alias("src_pid"), F.col("b.ID_ML_str_r").alias("dst_pid"))
    .dropDuplicates()
    .withColumn("rule_family", F.lit("R6_VENDOR_CLASS_CAT_DESC_SIM"))
)

# ============================================================================
# R7 — Vendor similar AND Desc similar AND Class similar (optional: same dept)
# ============================================================================
r7_left  = df_g.select("ID_ML_str","dept_key","vendor_tokens","desc_tokens","class_tokens")
r7_right = (r7_left
            .withColumnRenamed("ID_ML_str","ID_ML_str_r")
            .withColumnRenamed("dept_key","dept_key_r")
            .withColumnRenamed("vendor_tokens","vendor_tokens_r")
            .withColumnRenamed("desc_tokens","desc_tokens_r")
            .withColumnRenamed("class_tokens","class_tokens_r"))

r7_dept_cond = ((F.col("a.dept_key")==F.col("b.dept_key_r"))
                if R7_REQUIRE_SAME_DEPT else F.lit(True))

r7_pairs = (r7_left.alias("a")
    .join(r7_right.alias("b"),
          (F.col("a.ID_ML_str") < F.col("b.ID_ML_str_r")) & r7_dept_cond,
          "inner")
    .withColumn("vendor_jacc", jaccard_expr(F.col("a.vendor_tokens"), F.col("b.vendor_tokens_r")))
    .withColumn("desc_jacc",   jaccard_expr(F.col("a.desc_tokens"),   F.col("b.desc_tokens_r")))
    .withColumn("class_jacc",  jaccard_expr(F.col("a.class_tokens"),  F.col("b.class_tokens_r")))
    .withColumn("ok_sizes",
        (F.size("a.vendor_tokens") >= F.lit(MIN_TOKENS_VENDOR)) &
        (F.size("b.vendor_tokens_r")>=F.lit(MIN_TOKENS_VENDOR)) &
        (F.size("a.desc_tokens")   >= F.lit(MIN_TOKENS_DESC))   &
        (F.size("b.desc_tokens_r") >=F.lit(MIN_TOKENS_DESC))   &
        (F.size("a.class_tokens")  >= F.lit(MIN_TOKENS_CLASS))  &
        (F.size("b.class_tokens_r")>=F.lit(MIN_TOKENS_CLASS))
    )
    # AND semantics across all three similarities
    .filter(
        F.col("ok_sizes") &
        (F.col("vendor_jacc") >= F.lit(VENDOR_SIM_MIN_R7)) &
        (F.col("desc_jacc")   >= F.lit(DESC_SIM_MIN_R7))   &
        (F.col("class_jacc")  >= F.lit(CLASS_SIM_MIN_R7))
    )
    .select(F.col("a.ID_ML_str").alias("src_pid"), F.col("b.ID_ML_str_r").alias("dst_pid"))
    .dropDuplicates()
    .withColumn("rule_family", F.lit("R7_VENDOR_DESC_CLASS_SIM"))
)

# ----------------------------
# Final union of all rule families → edges_rules (undirected, deduped)
# ----------------------------
edges_all_new = (edges_r1
    .unionByName(edges_r2_exact, allowMissingColumns=True)
    .unionByName(edges_r2_sim,   allowMissingColumns=True)
    .unionByName(edges_r3_exact, allowMissingColumns=True)
    .unionByName(edges_r3_sim,   allowMissingColumns=True)
    .unionByName(r4_pairs,       allowMissingColumns=True)   # relaxed R4
    .unionByName(r5_pairs,       allowMissingColumns=True)
    .unionByName(r6_pairs,       allowMissingColumns=True)
    .unionByName(r7_pairs,       allowMissingColumns=True)
    .select(
        F.col("src_pid").cast("string").alias("src_pid"),
        F.col("dst_pid").cast("string").alias("dst_pid"),
        "rule_family"
    )
    .dropDuplicates()
)

edges_all = edges_all_new

edges_rules = (
    edges_all
      .withColumn("a", F.least(F.col("src_pid"), F.col("dst_pid")))
      .withColumn("b", F.greatest(F.col("src_pid"), F.col("dst_pid")))
      .select(F.col("a").alias("src_pid"), F.col("b").alias("dst_pid"), "rule_family")
      .dropDuplicates()
).cache()

print("\n[Step 2/extended + compact + R4 relaxed] Edge counts by rule_family:")
display(edges_rules.groupBy("rule_family").agg(F.count("*").alias("edge_count")).orderBy(F.desc("edge_count")))
print(f"[Step 2/extended + compact + R4 relaxed] Total unique edges emitted: {edges_rules.count():,}")


In [0]:

# STEP 2 (extended, FIXED + compact VSN, R4 relaxed) — Build PS-group edges
# This cell fully replaces Step 2.
# Changes vs. prior version:
#   - Adds vendor_style_compact (alphanumeric-only VSN) for exact VSN checks (R1, R5).
#   - R4 is RELAXED: NO vendor-name requirement. Edge if (VSN_sim >= thr AND DESC_sim >= thr).
#   - Keeps your AND-gated R5–R7 and exact/sim R2–R3.

from pyspark.sql import functions as F, types as T

# ----------------------------
# Thresholds & switches
# ----------------------------
# Existing similarity gates (exact rules are separate)
CLASS_SIM_JACCARD     = 0.40
CATEGORY_SIM_JACCARD  = 0.40

# R4 (RELAXED) — VSN similar AND Description similar (NO vendor-name anchor)
VSN_SIM_MIN_R4        = 0.30
DESC_SIM_MIN_R4       = 0.30

# R6 — same vendor + class + category AND Description similar
DESC_SIM_MIN_R6       = 0.35

# R7 — Vendor similar AND Description similar AND Class similar (optional: same dept)
VENDOR_SIM_MIN_R7     = 0.30
DESC_SIM_MIN_R7       = 0.25
CLASS_SIM_MIN_R7      = 0.30

# R8 — Description-only similarity (NO dept condition)
DESC_SIM_MIN_R8       = 0.35

# Minimum token counts to avoid spurious matches
MIN_TOKENS_VENDOR = 1
MIN_TOKENS_VSN    = 1
MIN_TOKENS_CLASS  = 1
MIN_TOKENS_DESC   = 2

# ----------------------------
# Helpers
# ----------------------------
def tokens_expr(colname):
    # distinct, non-empty word tokens
    return F.expr(
        f"filter(array_distinct(split(regexp_replace(coalesce({colname},''), '[^\\w]+', ' '), '\\\\s+')), x -> x <> '')"
    )

def jaccard_expr(a, b):
    # token-count guards ensure union > 0
    return (F.size(F.array_intersect(a, b)) / F.size(F.array_union(a, b)))

def star_edges_from_pids(df_groups, pids_col: str, rule_family: str):
    return (
        df_groups
          .withColumn("pids_sorted", F.array_sort(F.col(pids_col)))
          .withColumn("anchor_pid", F.element_at(F.col("pids_sorted"), 1))
          .withColumn("others", F.slice(F.col("pids_sorted"), 2, 1000000))
          .select("anchor_pid", F.explode_outer("others").alias("other_pid"))
          .filter(F.col("anchor_pid").isNotNull() & F.col("other_pid").isNotNull() & (F.col("anchor_pid") != F.col("other_pid")))
          .withColumn("src_pid", F.least(F.col("anchor_pid"), F.col("other_pid")))
          .withColumn("dst_pid", F.greatest(F.col("anchor_pid"), F.col("other_pid")))
          .select("src_pid","dst_pid").dropDuplicates()
          .withColumn("rule_family", F.lit(rule_family))
    )

# ----------------------------
# Preconditions & normalization
# ----------------------------
need = ["ID_ML","vendor_name_norm","vendor_style_norm","class_name_norm","category_name_norm"]
missing = [c for c in need if c not in df_compare_norm.columns]
if missing:
    raise ValueError(f"[Step 2] df_compare_norm missing columns: {missing}")

# Add desc_norm if missing
if "desc_norm" not in df_compare_norm.columns:
    df_compare_norm = df_compare_norm.withColumn(
        "desc_norm",
        F.trim(F.lower(F.regexp_replace(F.coalesce(F.col("Product Description (clean)"), F.lit("")), r"[^\w]+", " ")))
    )

# Add vendor_style_compact (lowercase alphanumeric-only) for exact VSN equality
df_compare_norm = df_compare_norm.withColumn(
    "vendor_style_compact",
    F.lower(F.regexp_replace(F.coalesce(F.col("vendor_style_norm"), F.lit("")), r"[^A-Za-z0-9]+", ""))
)

# Cast ID to string for graph nodes
df_base = df_compare_norm.withColumn("ID_ML_str", F.col("ID_ML").cast("string"))

# ----------------------------
# Build tokens & deduplicate per ID_ML
# ----------------------------
df_tok = (
    df_base
      .withColumn("vendor_tokens", tokens_expr("vendor_name_norm"))
      .withColumn("vsn_tokens",    tokens_expr("vendor_style_norm"))   # similarity uses token form
      .withColumn("class_tokens",  tokens_expr("class_name_norm"))
      .withColumn("cat_tokens",    tokens_expr("category_name_norm"))
      .withColumn("desc_tokens",   tokens_expr("desc_norm"))
      .withColumn("dept_key",      F.coalesce(F.col("dept_norm"), F.lit("__NO_DEPT__")))
)

cols_keep = ["ID_ML_str","vendor_name_norm","vendor_style_norm","vendor_style_compact",
             "class_name_norm","category_name_norm","dept_key",
             "vendor_tokens","vsn_tokens","class_tokens","cat_tokens","desc_tokens"]
df_g = df_tok.select(*cols_keep).dropDuplicates(["ID_ML_str"])

# ============================================================================
# R1 — EXACT: Same vendor style (using compact VSN)
# ============================================================================
r1_candidates = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & (F.col("vendor_style_compact") != ""))
      .select("vendor_style_compact","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_style_compact")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("pids") > 1)
)
edges_r1 = star_edges_from_pids(r1_candidates, "pids", "R1_VENDOR_STYLE")

# ============================================================================
# R2 — vendor + class_name: exact & similar
# ============================================================================
# Exact
r2a_candidates = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & F.col("class_name_norm").isNotNull())
      .select("vendor_name_norm","class_name_norm","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_name_norm","class_name_norm")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("pids") > 1)
)
edges_r2_exact = star_edges_from_pids(r2a_candidates, "pids", "R2_VENDOR_CLASS_EXACT")

# Similar
r2b_keys = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & F.col("class_name_norm").isNotNull())
      .select("vendor_name_norm","class_name_norm","class_tokens","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_name_norm","class_name_norm","class_tokens")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("class_tokens") > 0)
)
r2b_pairs = (
    r2b_keys.alias("a")
      .join(r2b_keys.alias("b"),
            (F.col("a.vendor_name_norm")==F.col("b.vendor_name_norm")) &
            (F.col("a.class_name_norm") < F.col("b.class_name_norm")),
            "inner")
      .withColumn("jacc", jaccard_expr(F.col("a.class_tokens"), F.col("b.class_tokens")))
      .filter(F.col("jacc") >= F.lit(CLASS_SIM_JACCARD))
      .select(F.array_distinct(F.array_union(F.col("a.pids"), F.col("b.pids"))).alias("pids_union"))
      .filter(F.size("pids_union") > 1)
)
edges_r2_sim = star_edges_from_pids(r2b_pairs.withColumnRenamed("pids_union","pids_union"), "pids_union", "R2_VENDOR_CLASS_SIM")

# ============================================================================
# R3 — vendor + category_name: exact & similar
# ============================================================================
# Exact
r3a_candidates = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & F.col("category_name_norm").isNotNull())
      .select("vendor_name_norm","category_name_norm","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_name_norm","category_name_norm")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("pids") > 1)
)
edges_r3_exact = star_edges_from_pids(r3a_candidates, "pids", "R3_VENDOR_CATEGORY_EXACT")

# Similar
r3b_keys = (
    df_g
      .filter(F.col("vendor_name_norm").isNotNull() & F.col("category_name_norm").isNotNull())
      .select("vendor_name_norm","category_name_norm","cat_tokens","ID_ML_str")
      .dropDuplicates()
      .groupBy("vendor_name_norm","category_name_norm","cat_tokens")
      .agg(F.collect_set("ID_ML_str").alias("pids"))
      .filter(F.size("cat_tokens") > 0)
)
r3b_pairs = (
    r3b_keys.alias("a")
      .join(r3b_keys.alias("b"),
            (F.col("a.vendor_name_norm")==F.col("b.vendor_name_norm")) &
            (F.col("a.category_name_norm") < F.col("b.category_name_norm")),
            "inner")
      .withColumn("jacc", jaccard_expr(F.col("a.cat_tokens"), F.col("b.cat_tokens")))
      .filter(F.col("jacc") >= F.lit(CATEGORY_SIM_JACCARD))
      .select(F.array_distinct(F.array_union(F.col("a.pids"), F.col("b.pids"))).alias("pids_union"))
      .filter(F.size("pids_union") > 1)
)
edges_r3_sim = star_edges_from_pids(r3b_pairs.withColumnRenamed("pids_union","pids_union"), "pids_union", "R3_VENDOR_CATEGORY_SIM")

# ============================================================================
# R4 (RELAXED) — VSN similar AND Desc similar → edge  (NO vendor-name requirement)
# ============================================================================
r4_left  = df_g.select("ID_ML_str","vsn_tokens","desc_tokens")
r4_right = (r4_left
            .withColumnRenamed("ID_ML_str","ID_ML_str_r")
            .withColumnRenamed("vsn_tokens","vsn_tokens_r")
            .withColumnRenamed("desc_tokens","desc_tokens_r"))

r4_pairs = (r4_left.alias("a")
    .join(r4_right.alias("b"),
          (F.col("a.ID_ML_str") < F.col("b.ID_ML_str_r")),
          "inner")
    .withColumn("vsn_jacc",  jaccard_expr(F.col("a.vsn_tokens"),  F.col("b.vsn_tokens_r")))
    .withColumn("desc_jacc", jaccard_expr(F.col("a.desc_tokens"), F.col("b.desc_tokens_r")))
    .withColumn("ok_sizes",
        (F.size("a.vsn_tokens")  >= F.lit(MIN_TOKENS_VSN))  &
        (F.size("b.vsn_tokens_r")>= F.lit(MIN_TOKENS_VSN))  &
        (F.size("a.desc_tokens") >= F.lit(MIN_TOKENS_DESC)) &
        (F.size("b.desc_tokens_r")>=F.lit(MIN_TOKENS_DESC))
    )
    # AND semantics: both similarities must pass
    .filter(F.col("ok_sizes") &
            (F.col("vsn_jacc")  >= F.lit(VSN_SIM_MIN_R4)) &
            (F.col("desc_jacc") >= F.lit(DESC_SIM_MIN_R4)))
    .select(F.col("a.ID_ML_str").alias("src_pid"), F.col("b.ID_ML_str_r").alias("dst_pid"))
    .dropDuplicates()
    .withColumn("rule_family", F.lit("R4_VSN_DESC_SIM_NO_VENDOR"))
)



# ============================================================================
# R6 — same vendor + class + category AND Desc similar
# ============================================================================
r6_left  = df_g.select("vendor_name_norm","class_name_norm","category_name_norm","ID_ML_str","desc_tokens")
r6_right = (r6_left
            .withColumnRenamed("ID_ML_str","ID_ML_str_r")
            .withColumnRenamed("desc_tokens","desc_tokens_r"))

r6_pairs = (r6_left.alias("a")
    .join(r6_right.alias("b"),
          (F.col("a.vendor_name_norm")==F.col("b.vendor_name_norm")) &
          (F.col("a.class_name_norm")==F.col("b.class_name_norm")) &
          (F.col("a.category_name_norm")==F.col("b.category_name_norm")) &
          (F.col("a.ID_ML_str") < F.col("b.ID_ML_str_r")),
          "inner")
    .withColumn("desc_jacc", jaccard_expr(F.col("a.desc_tokens"), F.col("b.desc_tokens_r")))
    .withColumn("ok_sizes",
        (F.size("a.desc_tokens") >= F.lit(MIN_TOKENS_DESC)) &
        (F.size("b.desc_tokens_r")>=F.lit(MIN_TOKENS_DESC))
    )
    .filter(F.col("ok_sizes") & (F.col("desc_jacc") >= F.lit(DESC_SIM_MIN_R6)))
    .select(F.col("a.ID_ML_str").alias("src_pid"), F.col("b.ID_ML_str_r").alias("dst_pid"))
    .dropDuplicates()
    .withColumn("rule_family", F.lit("R6_VENDOR_CLASS_CAT_DESC_SIM"))
)

# ============================================================================
# R7 — Vendor similar AND Desc similar AND Class similar (optional: same dept)
# ============================================================================
r7_left  = df_g.select("ID_ML_str","dept_key","vendor_tokens","desc_tokens","class_tokens")
r7_right = (r7_left
            .withColumnRenamed("ID_ML_str","ID_ML_str_r")
            .withColumnRenamed("dept_key","dept_key_r")
            .withColumnRenamed("vendor_tokens","vendor_tokens_r")
            .withColumnRenamed("desc_tokens","desc_tokens_r")
            .withColumnRenamed("class_tokens","class_tokens_r"))

r7_pairs = (r7_left.alias("a")
    .join(r7_right.alias("b"),
          (F.col("a.ID_ML_str") < F.col("b.ID_ML_str_r")),
          "inner")
    .withColumn("vendor_jacc", jaccard_expr(F.col("a.vendor_tokens"), F.col("b.vendor_tokens_r")))
    .withColumn("desc_jacc",   jaccard_expr(F.col("a.desc_tokens"),   F.col("b.desc_tokens_r")))
    .withColumn("class_jacc",  jaccard_expr(F.col("a.class_tokens"),  F.col("b.class_tokens_r")))
    .withColumn("ok_sizes",
        (F.size("a.vendor_tokens") >= F.lit(MIN_TOKENS_VENDOR)) &
        (F.size("b.vendor_tokens_r")>=F.lit(MIN_TOKENS_VENDOR)) &
        (F.size("a.desc_tokens")   >= F.lit(MIN_TOKENS_DESC))   &
        (F.size("b.desc_tokens_r") >=F.lit(MIN_TOKENS_DESC))   &
        (F.size("a.class_tokens")  >= F.lit(MIN_TOKENS_CLASS))  &
        (F.size("b.class_tokens_r")>=F.lit(MIN_TOKENS_CLASS))
    )
    # AND semantics across all three similarities
    .filter(
        F.col("ok_sizes") &
        (F.col("vendor_jacc") >= F.lit(VENDOR_SIM_MIN_R7)) &
        (F.col("desc_jacc")   >= F.lit(DESC_SIM_MIN_R7))   &
        (F.col("class_jacc")  >= F.lit(CLASS_SIM_MIN_R7))
    )
    .select(F.col("a.ID_ML_str").alias("src_pid"), F.col("b.ID_ML_str_r").alias("dst_pid"))
    .dropDuplicates()
    .withColumn("rule_family", F.lit("R7_VENDOR_DESC_CLASS_SIM"))
)

# ============================================================================
# R8 — Description-only similarity, desc_jacc >= 0.35 (NO dept condition)
# ============================================================================
r8_left  = df_g.select("ID_ML_str","desc_tokens")
r8_right = (r8_left
            .withColumnRenamed("ID_ML_str","ID_ML_str_r")
            .withColumnRenamed("desc_tokens","desc_tokens_r"))

r8_pairs = (r8_left.alias("a")
    .join(r8_right.alias("b"),
          (F.col("a.ID_ML_str") < F.col("b.ID_ML_str_r")),
          "inner")
    .withColumn("desc_jacc", jaccard_expr(F.col("a.desc_tokens"), F.col("b.desc_tokens_r")))
    .withColumn("ok_sizes",
        (F.size("a.desc_tokens") >= F.lit(MIN_TOKENS_DESC)) &
        (F.size("b.desc_tokens_r")>=F.lit(MIN_TOKENS_DESC))
    )
    .filter(F.col("ok_sizes") & (F.col("desc_jacc") >= F.lit(DESC_SIM_MIN_R8)))
    .select(F.col("a.ID_ML_str").alias("src_pid"), F.col("b.ID_ML_str_r").alias("dst_pid"))
    .dropDuplicates()
    .withColumn("rule_family", F.lit("R8_DESC_SIM"))
)

# ----------------------------
# Final union of all rule families → edges_rules (undirected, deduped)
# ----------------------------
edges_all_new = (edges_r1
    .unionByName(edges_r2_exact, allowMissingColumns=True)
    .unionByName(edges_r2_sim,   allowMissingColumns=True)
    .unionByName(edges_r3_exact, allowMissingColumns=True)
    .unionByName(edges_r3_sim,   allowMissingColumns=True)
    .unionByName(r4_pairs,       allowMissingColumns=True)
    .unionByName(r6_pairs,       allowMissingColumns=True)
    .unionByName(r7_pairs,       allowMissingColumns=True)
    .unionByName(r8_pairs,       allowMissingColumns=True)   # R8 desc-only
    .select(
        F.col("src_pid").cast("string").alias("src_pid"),
        F.col("dst_pid").cast("string").alias("dst_pid"),
        "rule_family"
    )
    .dropDuplicates()
)

edges_all = edges_all_new

edges_rules = (
    edges_all
      .withColumn("a", F.least(F.col("src_pid"), F.col("dst_pid")))
      .withColumn("b", F.greatest(F.col("src_pid"), F.col("dst_pid")))
      .select(F.col("a").alias("src_pid"), F.col("b").alias("dst_pid"), "rule_family")
      .dropDuplicates()
).cache()

print("\nStep 2 with R1-R8, R5 exlcuded because redundant")
display(edges_rules.groupBy("rule_family").agg(F.count("*").alias("edge_count")).orderBy(F.desc("edge_count")))
print(f"[\nStep 2 with R1-R8, R5 exlcuded because redundant] Total unique edges emitted: {edges_rules.count():,}")



### Step 3 (Notebook‑only fallback) — Union–Find Connected Components → Stable `PS_PRODUCT_ID_NEW`

**What this cell does**  
- Collects the **ID_ML graph** from `edges_rules` (Step 2) and **computes connected components** on the **driver** using a fast Union–Find (Disjoint Set) algorithm.  
- Uses the **minimum `ID_ML` string** in each component as the stable **`PS_PRODUCT_ID_NEW`** (prevents ID churn).  
- Broadcasts the mapping back to Spark, **joins** it to `df_compare_norm`, and stamps `RULESET_VERSION`.  
- Produces `pid_to_new_str` in the same shape Step 4 expects.

**Why this approach**  
- **Notebook‑only** (no cluster libraries or GraphFrames JAR required).  
- Efficient for your graph size (~58k vertices, ~80k edges).  
- Identical downstream behavior to the GraphFrames plan: Step 4 **Audit** will work as is.

**Inputs**  
- `df_compare_norm` (Step 1)  
- `edges_rules` (Step 2; columns `src_pid`, `dst_pid`, `rule_family`)

**Outputs**  
- `df_final_new` with **`PS_PRODUCT_ID_NEW`** and `RULESET_VERSION`  
- `pid_to_new_str`: mapping of (`pid` = `ID_ML_str`, `comp_id` = `PS_PRODUCT_ID_NEW`) for Step 4  
- Diagnostics: component counts & largest components


In [0]:

# STEP 3 (Notebook-only) — Union–Find connected components over ID_ML graph

from pyspark.sql import functions as F, types as T
import sys

RULESET_VERSION = "v1.0"

# -------------------------------
# 1) Build vertex set and edge list
# -------------------------------
# Vertices from rows
nodes_from_rows = (
    df_compare_norm
      .select(F.col("ID_ML").cast("string").alias("pid"))
      .dropDuplicates()
)

# Vertices implied by edges (safe-guard)
nodes_from_edges = (
    edges_rules
      .select(F.col("src_pid").cast("string").alias("pid"))
      .unionByName(edges_rules.select(F.col("dst_pid").cast("string").alias("pid")))
      .dropDuplicates()
)

nodes_df = nodes_from_rows.unionByName(nodes_from_edges).dropDuplicates()

# Collect vertices & edges to the driver (fits easily at ~58k nodes / ~80k edges)
vertices = [r.pid for r in nodes_df.collect()]
edges = [(r.src_pid, r.dst_pid) for r in edges_rules.select(
            F.col("src_pid").cast("string").alias("src_pid"),
            F.col("dst_pid").cast("string").alias("dst_pid")
         ).dropDuplicates().collect()]

print(f"[Step 3/UF] Collected vertices: {len(vertices):,} ; edges: {len(edges):,}")

# -------------------------------
# 2) Union–Find (Disjoint Set) on driver
# -------------------------------
sys.setrecursionlimit(max(10000, len(vertices) * 2))

parent = {}
rank = {}

def find(x):
    # path compression
    parent.setdefault(x, x)
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return
    ra_rank = rank.get(ra, 0)
    rb_rank = rank.get(rb, 0)
    if ra_rank < rb_rank:
        parent[ra] = rb
    elif ra_rank > rb_rank:
        parent[rb] = ra
    else:
        parent[rb] = ra
        rank[ra] = ra_rank + 1

# Initialize parents
for v in vertices:
    parent[v] = v

# Union edges (undirected)
for a, b in edges:
    if a is None or b is None:
        continue
    if a == b:
        continue
    union(a, b)

# Build components: root -> members
components = {}
for v in vertices:
    r = find(v)
    components.setdefault(r, []).append(v)

# Canonical id per component = min(ID_ML_str)
root_to_canon = {root: min(members) for root, members in components.items()}

# Mapping node -> canonical id
mapping = [(v, root_to_canon[find(v)]) for v in vertices]

# -------------------------------
# 3) Create mapping DF and join back to rows
# -------------------------------
pid_to_new = spark.createDataFrame(mapping, schema=T.StructType([
    T.StructField("ID_ML_str", T.StringType(), False),
    T.StructField("PS_PRODUCT_ID_NEW", T.StringType(), False),
]))

n_components = len(set(root_to_canon.values()))
print(f"[Step 3/UF] Components (final groups) formed: {n_components:,}")

# Join to rows and stamp version
df_final_new = (
    df_compare_norm
      .withColumn("ID_ML_str", F.col("ID_ML").cast("string"))
      .join(pid_to_new, on="ID_ML_str", how="left")
      .drop("ID_ML_str")
      .withColumn("RULESET_VERSION", F.lit(RULESET_VERSION))
)

# Safety backfill for any missing (shouldn't occur)
df_final_new = df_final_new.withColumn(
    "PS_PRODUCT_ID_NEW",
    F.coalesce(F.col("PS_PRODUCT_ID_NEW"), F.col("ID_ML").cast("string"))
)

# -------------------------------
# 4) Diagnostics (largest components)
# -------------------------------
comp_sizes = (
    df_final_new.groupBy("PS_PRODUCT_ID_NEW")
                .agg(F.countDistinct("rec_id").alias("rows_in_component"))
)
print("[Step 3/UF] Top 10 largest components (rows):")
display(comp_sizes.orderBy(F.desc("rows_in_component")).limit(10))

MEGA_THRESHOLD = 2000
mega_count = comp_sizes.filter(F.col("rows_in_component") > MEGA_THRESHOLD).count()
if mega_count > 0:
    print(f"[Step 3/UF][WARN] {mega_count} components exceed {MEGA_THRESHOLD} rows. Review thresholds or rules.")

print("[Step 3/UF] df_final_new is ready with PS_PRODUCT_ID_NEW.")
display(df_final_new.select("rec_id", "ID_ML", "PS_PRODUCT_ID_NEW").limit(20))

# -------------------------------
# 5) Prepare mapping for Step 4 (Audit) in the expected shape
# -------------------------------
pid_to_new_str = spark.createDataFrame(
    [(pid, comp) for pid, comp in mapping],
    schema=T.StructType([
        T.StructField("pid", T.StringType(), False),
        T.StructField("comp_id", T.StringType(), False),
    ])
)


In [0]:

# Enable Arrow for faster conversion (optional but recommended)
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# Convert to pandas
df_final_new_pd = df_final_new.toPandas()


In [0]:
df_final_new_pd.head(5)

In [0]:
df_final_new_pd.to_csv("/dbfs/FileStore/GPSTest1.csv", index=False, encoding = "utf-8-sig")


# Getting the workspace URL

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")  


# File path relative to /dbfs/FileStore

file_name = "GPSTest1.csv"

download_url = f"https://{workspace_url}/files/{file_name}"

print("Download link:", download_url)
 


### Step 4 — Audit & Provenance Roll‑up (`RULE_KEYS_HIT`) + Impact Summary

**What this cell does**  
- Rolls up **which hard‑rule families** actually caused merges for each final component and attaches that as **`RULE_KEYS_HIT`** to every row.  
- Builds an **audit summary** per component: number of model groups (`ID_ML`) merged, number of rows, and **top vendors / classes / categories**.  
- Prints an **impact summary** (how many rows and `ID_ML` groups changed due to rules) and resurfaces **edge counts by rule family** for quick context.

**Why it matters (business)**  
- Gives Product Compliance and stakeholders a transparent view of *why* records were merged across model groups—e.g., “Vendor+Class (exact)” or “Vendor+Category (similar)”.  
- Supports audit trails and PBI tooltips/explainers, reducing back‑and‑forth during reviews.

**How it works (technical)**  
- Uses the mapping from Step 3 (`pid_to_new`: `ID_ML_str` → `PS_PRODUCT_ID_NEW`) to attribute every **edge** in `edges_rules` to the **component** it belongs to, then **collects** distinct `rule_family` per component.  
- Joins this roll‑up back to **`df_final_new`**, producing **`df_final_new_audit`** with the new `RULE_KEYS_HIT` column.  
- Computes **component‑level** and **pipeline‑level** diagnostics (largest components, % changed, etc.).  

**Inputs expected**  
- `df_final_new` (from Step 3)  
- `pid_to_new` (from Step 3; columns: `ID_ML_str`, `PS_PRODUCT_ID_NEW`)  
- `edges_rules` (from Step 2; columns: `src_pid`, `dst_pid`, `rule_family`)  

**Outputs**  
- `df_final_new_audit`: `df_final_new` **plus** `RULE_KEYS_HIT` (array\<string>) and `RULESET_VERSION` (already present).  
- `comp_audit`: component‑level summary for review & export if needed.  
- Printed impact metrics (changed vs unchanged).


In [0]:

# STEP 4 (pandas) — Reasons for why a line moved to PS_PRODUCT_ID_NEW

import pandas as pd
import numpy as np

# -----------------------------
# 0) Pull edges_rules to pandas
# -----------------------------
edges_rules_pd = None
if 'edges_rules' in globals():
    try:
        # If it's a Spark DF, convert to pandas
        from pyspark.sql import DataFrame as SparkDataFrame  # type: ignore
        if isinstance(edges_rules, SparkDataFrame):
            edges_rules_pd = edges_rules.select('src_pid','dst_pid','rule_family').toPandas()
        else:
            # Assume it's already a pandas-like object
            edges_rules_pd = edges_rules[['src_pid','dst_pid','rule_family']].copy()
    except Exception:
        # Fallback attempt: try attribute access
        try:
            edges_rules_pd = edges_rules[['src_pid','dst_pid','rule_family']].copy()
        except Exception as e:
            raise ValueError(f"[Step 4/pandas] Couldn't access edges_rules columns: {e}")
else:
    raise ValueError("[Step 4/pandas] 'edges_rules' not found in session; needed to explain why groups were merged.")

# -----------------------------
# 1) Normalization helpers for IDs (string-safe)
# -----------------------------
def _norm_pid(x):
    """Return a stable string id from int/float/str (handles 12425.0 -> '12425')."""
    if pd.isna(x):
        return None
    try:
        xf = float(x)
        if np.isfinite(xf) and xf.is_integer():
            return str(int(xf))
    except Exception:
        pass
    s = str(x).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

# Ensure df_final_new_pd has normalized ID strings
if 'ID_ML' not in df_final_new_pd.columns or 'PS_PRODUCT_ID_NEW' not in df_final_new_pd.columns:
    raise ValueError("[Step 4/pandas] df_final_new_pd must include 'ID_ML' and 'PS_PRODUCT_ID_NEW'.")

df_final_new_pd = df_final_new_pd.copy()
df_final_new_pd['ID_ML_str'] = df_final_new_pd['ID_ML'].map(_norm_pid)
df_final_new_pd['PS_PRODUCT_ID_NEW'] = df_final_new_pd['PS_PRODUCT_ID_NEW'].map(_norm_pid)  # force string

# Normalize edges
edges_rules_pd = edges_rules_pd.copy()
edges_rules_pd['src_pid'] = edges_rules_pd['src_pid'].map(_norm_pid)
edges_rules_pd['dst_pid'] = edges_rules_pd['dst_pid'].map(_norm_pid)
edges_rules_pd = edges_rules_pd.dropna(subset=['src_pid','dst_pid']).drop_duplicates()

# -----------------------------------------
# 2) Component map: ID_ML_str -> component
# -----------------------------------------
comp_map = dict(zip(df_final_new_pd['ID_ML_str'], df_final_new_pd['PS_PRODUCT_ID_NEW']))

# Attach component to edges (both ends), then keep edges that lie fully inside a single component
edges_rules_pd['comp_src'] = edges_rules_pd['src_pid'].map(comp_map)
edges_rules_pd['comp_dst'] = edges_rules_pd['dst_pid'].map(comp_map)
edges_in_comp = edges_rules_pd.dropna(subset=['comp_src','comp_dst']).copy()
edges_in_comp = edges_in_comp.loc[edges_in_comp['comp_src'] == edges_in_comp['comp_dst']].copy()
edges_in_comp.rename(columns={'comp_src':'component'}, inplace=True)
edges_in_comp.drop(columns=['comp_dst'], inplace=True)

# ---------------------------------------------------
# 3) Roll-ups:
#    (A) RULE_KEYS_HIT per component (all edges inside)
#    (B) RULE_KEYS_HIT_IDML per ID_ML (direct edges at node)
# ---------------------------------------------------
# (A) component-level unique rule families
comp_rules = (
    edges_in_comp
      .groupby('component')['rule_family']
      .agg(lambda s: sorted(set(s)))
      .to_dict()
)

# (B) idml-level unique rule families (collect src & dst)
idml_rules_src = edges_in_comp[['src_pid','rule_family']].rename(columns={'src_pid':'pid'}).drop_duplicates()
idml_rules_dst = edges_in_comp[['dst_pid','rule_family']].rename(columns={'dst_pid':'pid'}).drop_duplicates()
idml_rules_all = pd.concat([idml_rules_src, idml_rules_dst], ignore_index=True).drop_duplicates()
idml_rules = (
    idml_rules_all
      .groupby('pid')['rule_family']
      .agg(lambda s: sorted(set(s)))
      .to_dict()
)

# -----------------------------------------
# 4) Attach reasons back to each line (row)
# -----------------------------------------
df_final_new_pd['RULE_KEYS_HIT'] = df_final_new_pd['PS_PRODUCT_ID_NEW'].map(comp_rules)
df_final_new_pd['RULE_KEYS_HIT'] = df_final_new_pd['RULE_KEYS_HIT'].apply(lambda x: x if isinstance(x, list) else [])

df_final_new_pd['RULE_KEYS_HIT_IDML'] = df_final_new_pd['ID_ML_str'].map(idml_rules)
df_final_new_pd['RULE_KEYS_HIT_IDML'] = df_final_new_pd['RULE_KEYS_HIT_IDML'].apply(lambda x: x if isinstance(x, list) else [])

# Human-readable WHY_MERGED
def _why(row):
    id_ml = row['ID_ML_str']
    comp  = row['PS_PRODUCT_ID_NEW']
    moved = (id_ml != comp)
    comp_rules_list = row['RULE_KEYS_HIT'] or []
    idml_rules_list = row['RULE_KEYS_HIT_IDML'] or []

    if not moved:
        if comp_rules_list:
            return f"Unchanged: in a component with edges {', '.join(comp_rules_list)} (no direct edge at this ID_ML)."
        return "Unchanged (no hard-rule impact)."
    # moved
    if idml_rules_list:
        return f"Moved via direct edge(s) at this ID_ML: {', '.join(idml_rules_list)}."
    if comp_rules_list:
        return f"Moved indirectly (transitive) via component edges: {', '.join(comp_rules_list)}."
    return "Moved, but no rule edges were recorded for this component (check inputs)."

df_final_new_pd['WHY_MERGED'] = df_final_new_pd.apply(_why, axis=1)

print("[Step 4/pandas] Added RULE_KEYS_HIT, RULE_KEYS_HIT_IDML, WHY_MERGED.")
display(df_final_new_pd[['rec_id','ID_ML_str','PS_PRODUCT_ID_NEW','RULE_KEYS_HIT_IDML','RULE_KEYS_HIT','WHY_MERGED']].head(20))

# -----------------------------------------------------------
# 5) Helper: explain one line by rec_id with example edges
# -----------------------------------------------------------
def explain_line(rec_id, max_examples=10):
    """
    Print the reason(s) for a specific line and show a few example edges that
    involve the line's ID_ML within the same component.
    """
    sub = df_final_new_pd.loc[df_final_new_pd['rec_id'] == rec_id]
    if sub.empty:
        print(f"[explain_line] rec_id {rec_id} not found.")
        return
    r = sub.iloc[0]
    pid = r['ID_ML_str']
    comp = r['PS_PRODUCT_ID_NEW']

    print(f"\n--- Explain rec_id={rec_id} ---")
    print(f"ID_ML={pid}  →  PS_PRODUCT_ID_NEW={comp}")
    print(f"WHY_MERGED: {r['WHY_MERGED']}")
    print(f"Component-level rules: {r['RULE_KEYS_HIT']}")
    print(f"Direct ID_ML rules:    {r['RULE_KEYS_HIT_IDML']}")

    # Show example edges inside this component that touch this ID_ML
    ex = edges_in_comp.loc[
        ((edges_in_comp['src_pid'] == pid) | (edges_in_comp['dst_pid'] == pid))
        & (edges_in_comp['component'] == comp)
    ][['src_pid','dst_pid','rule_family']].drop_duplicates().head(max_examples)

    if ex.empty:
        print("(No direct edges at this ID_ML; merge may be via transitive connections.)")
    else:
        print("\nExample edges involving this ID_ML within the component:")
        display(ex)

# Example:
# explain_line(rec_id=12345)


In [0]:
df_final_new_pd.to_csv("/dbfs/FileStore/GPSTest6.csv", index=False, encoding = "utf-8-sig")


# Getting the workspace URL

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")  


# File path relative to /dbfs/FileStore

file_name = "GPSTest6.csv"

download_url = f"https://{workspace_url}/files/{file_name}"

print("Download link:", download_url)
 